In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# PROJECT PERSISTENCE SETUP
# ============================================================

import os
import json
from datetime import datetime

PROJECT_DIR = "/kaggle/working/skin_cancer_project"

CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
DATA_DIR       = os.path.join(PROJECT_DIR, "data")
RESULTS_DIR    = os.path.join(PROJECT_DIR, "results")
BACKUP_DIR     = os.path.join(PROJECT_DIR, "backups")

for folder in [
    PROJECT_DIR,
    CHECKPOINT_DIR,
    DATA_DIR,
    RESULTS_DIR,
    BACKUP_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("Project directories ready.")
print("PROJECT_DIR:", PROJECT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("BACKUP_DIR:", BACKUP_DIR)

In [1]:
# STEP 1 — Project Configuration and Safe Storage Setup

import os
import random
import numpy as np
import pandas as pd
import torch

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Project folders
PROJECT_DIR = "/kaggle/working/skin_cancer_project"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
DATA_DIR = os.path.join(PROJECT_DIR, "data")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Final cancer classes
CLASS_NAMES = [
    "Melanoma",
    "BCC",
    "SCC",
    "MCC"
]

NUM_CLASSES = len(CLASS_NAMES)

# Dataset paths
HAM_ROOT = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"

ISIC_ROOT = "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629"

DERMA_ROOT = "/kaggle/input/datasets/hiro002/dermacon-in-d-dataset"

MCC_ROOT = "/kaggle/input/datasets/quantumcoders03/datasetmcc"

MCC_NEW_ROOTS = [
    "/kaggle/input/datasets/quantumcoders03/mcc-1-dataset",
    "/kaggle/input/datasets/quantumcoders03/mcc-2-datatset",
    "/kaggle/input/datasets/quantumcoders03/mcc-3-datatse"
]

print("=" * 60)
print("PROJECT SETUP")
print("=" * 60)

print("Project directory :", PROJECT_DIR)
print("Checkpoint folder :", CHECKPOINT_DIR)
print("Data folder       :", DATA_DIR)
print("Results folder    :", RESULTS_DIR)

print("\nClasses:")
for i, name in enumerate(CLASS_NAMES):
    print(f"{i} -> {name}")

print("\nDataset paths:")
print("HAM10000 :", os.path.exists(HAM_ROOT))
print("ISIC     :", os.path.exists(ISIC_ROOT))
print("DermaCon :", os.path.exists(DERMA_ROOT))
print("MCC      :", os.path.exists(MCC_ROOT))

for i, path in enumerate(MCC_NEW_ROOTS, 1):
    print(f"MCC-{i}    :", os.path.exists(path))

print("\nPyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("Seed           :", SEED)

PROJECT SETUP
Project directory : /kaggle/working/skin_cancer_project
Checkpoint folder : /kaggle/working/skin_cancer_project/checkpoints
Data folder       : /kaggle/working/skin_cancer_project/data
Results folder    : /kaggle/working/skin_cancer_project/results

Classes:
0 -> Melanoma
1 -> BCC
2 -> SCC
3 -> MCC

Dataset paths:
HAM10000 : True
ISIC     : True
DermaCon : False
MCC      : False
MCC-1    : False
MCC-2    : False
MCC-3    : False

PyTorch version: 2.10.0+cu128
CUDA available : True
Seed           : 42


In [3]:
# ============================================================
# PROJECT PERSISTENCE SETUP
# ============================================================

import os
import json
from datetime import datetime

PROJECT_DIR = "/kaggle/working/skin_cancer_project"

CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
DATA_DIR       = os.path.join(PROJECT_DIR, "data")
RESULTS_DIR    = os.path.join(PROJECT_DIR, "results")
BACKUP_DIR     = os.path.join(PROJECT_DIR, "backups")

for folder in [
    PROJECT_DIR,
    CHECKPOINT_DIR,
    DATA_DIR,
    RESULTS_DIR,
    BACKUP_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("Project directories ready.")
print("PROJECT_DIR:", PROJECT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("BACKUP_DIR:", BACKUP_DIR)

Project directories ready.
PROJECT_DIR: /kaggle/working/skin_cancer_project
CHECKPOINT_DIR: /kaggle/working/skin_cancer_project/checkpoints
DATA_DIR: /kaggle/working/skin_cancer_project/data
RESULTS_DIR: /kaggle/working/skin_cancer_project/results
BACKUP_DIR: /kaggle/working/skin_cancer_project/backups


In [4]:
# STEP 2 — Verify Dataset Paths and Folder Structure

import os

print("=" * 70)
print("DATASET STRUCTURE CHECK")
print("=" * 70)

# Show all available dataset folders
datasets_root = "/kaggle/input/datasets"

print("\nAvailable dataset folders:")
for item in sorted(os.listdir(datasets_root)):
    print(" -", item)

# Search for DermaCon-related folders
print("\n" + "=" * 70)
print("DERMACON SEARCH")
print("=" * 70)

for root, dirs, files in os.walk(datasets_root):
    root_lower = root.lower()

    if "derma" in root_lower or "dermacon" in root_lower:
        print("\nFolder:", root)
        print("Subfolders:", dirs[:10])
        print("Files:", files[:10])

# Check the other important datasets
print("\n" + "=" * 70)
print("IMPORTANT DATASET CHECK")
print("=" * 70)

paths_to_check = {
    "HAM10000": HAM_ROOT,
    "ISIC": ISIC_ROOT,
    "MCC": MCC_ROOT,
    "MCC-1": MCC_NEW_ROOTS[0],
    "MCC-2": MCC_NEW_ROOTS[1],
    "MCC-3": MCC_NEW_ROOTS[2]
}

for name, path in paths_to_check.items():
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        try:
            items = os.listdir(path)
            print("Items:", items[:10])
        except Exception as e:
            print("Error:", e)

DATASET STRUCTURE CHECK

Available dataset folders:
 - ahmedxc4
 - hiro002
 - kmader
 - nodoubttome
 - quantumcoders07
 - riyaelizashaju
 - tomooinubushi

DERMACON SEARCH

Folder: /kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/dermatofibroma
Subfolders: []
Files: ['ISIC_0024845.jpg', 'ISIC_0011865.jpg', 'ISIC_0001114.jpg', 'ISIC_0011478.jpg', 'ISIC_0011677.jpg', 'ISIC_0024330.jpg', 'ISIC_0024994.jpg', 'ISIC_0024973.jpg', 'ISIC_0011410.jpg', 'ISIC_0024396.jpg']

Folder: /kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Train/dermatofibroma
Subfolders: []
Files: ['ISIC_0027107.jpg', 'ISIC_0029973.jpg', 'ISIC_0025954.jpg', 'ISIC_0027216.jpg', 'ISIC_0029297.jpg', 'ISIC_0032468.jpg', 'ISIC_0031457.jpg', 'ISIC_0033554.jpg', 'ISIC_0029578.jpg', 'ISIC_0031271.jpg']

Folder: /kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/skin cancer

In [ ]:
# STEP 3 — Load and Inspect Main Dataset Metadata

import os
import pandas as pd

print("=" * 70)
print("STEP 3 — METADATA INSPECTION")
print("=" * 70)

# ---------------------------------------------------------
# 1. Correct DermaCon-IN path
# ---------------------------------------------------------

DERMA_ROOT = "/kaggle/input/datasets/hiro002/dermacon-in-dataset"

print("\nDermaCon-IN path:")
print(DERMA_ROOT)
print("Exists:", os.path.exists(DERMA_ROOT))

# ---------------------------------------------------------
# 2. HAM10000 metadata
# ---------------------------------------------------------

HAM_METADATA_PATH = os.path.join(
    HAM_ROOT,
    "HAM10000_metadata.csv"
)

ham = pd.read_csv(HAM_METADATA_PATH)

print("\n" + "-" * 70)
print("HAM10000")
print("-" * 70)

print("Shape:", ham.shape)
print("Columns:")
print(ham.columns.tolist())

print("\nDiagnosis distribution:")
print(ham["dx"].value_counts())

# ---------------------------------------------------------
# 3. ISIC 2024 metadata
# ---------------------------------------------------------

ISIC_METADATA_PATH = os.path.join(
    ISIC_ROOT,
    "metadata.csv"
)

isic = pd.read_csv(
    ISIC_METADATA_PATH,
    low_memory=False
)

print("\n" + "-" * 70)
print("ISIC 2024")
print("-" * 70)

print("Shape:", isic.shape)
print("Columns:")
print(isic.columns.tolist())

print("\nDiagnosis distribution — top 20:")
print(isic["diagnosis"].value_counts().head(20))

# ---------------------------------------------------------
# 4. DermaCon-IN metadata
# ---------------------------------------------------------

DERMA_METADATA_PATH = os.path.join(
    DERMA_ROOT,
    "METADATA",
    "Skin_Metadata.csv"
)

derma = pd.read_csv(DERMA_METADATA_PATH)

print("\n" + "-" * 70)
print("DermaCon-IN")
print("-" * 70)

print("Shape:", derma.shape)
print("Columns:")
print(derma.columns.tolist())

print("\nDisease distribution:")
print(derma["Disease_label"].value_counts().head(20))

# ---------------------------------------------------------
# 5. Candidate SCC dataset inspection
# ---------------------------------------------------------

SCC_CANDIDATE_ROOT = "/kaggle/input/datasets/ahmedxc4/skin-ds"

print("\n" + "-" * 70)
print("SCC CANDIDATE DATASET")
print("-" * 70)

print("Path:", SCC_CANDIDATE_ROOT)
print("Exists:", os.path.exists(SCC_CANDIDATE_ROOT))

if os.path.exists(SCC_CANDIDATE_ROOT):

    for split in ["train", "val", "test"]:

        split_path = os.path.join(
            SCC_CANDIDATE_ROOT,
            split
        )

        if os.path.exists(split_path):

            print(f"\n{split.upper()} folders:")

            folders = sorted([
                x for x in os.listdir(split_path)
                if os.path.isdir(os.path.join(split_path, x))
            ])

            for folder in folders:
                folder_path = os.path.join(
                    split_path,
                    folder
                )

                image_count = sum(
                    1
                    for f in os.listdir(folder_path)
                    if f.lower().endswith(
                        (".jpg", ".jpeg", ".png")
                    )
                )

                print(f"  {folder}: {image_count}")

print("\n" + "=" * 70)
print("STEP 3 COMPLETE")
print("=" * 70)

In [ ]:
# STEP 4 — HAM10000 Cancer Image Records

import os
import pandas as pd

print("=" * 70)
print("STEP 4 — HAM10000 PREPROCESSING")
print("=" * 70)

# ---------------------------------------------------------
# 1. HAM10000 metadata
# ---------------------------------------------------------

ham_mapping = {
    "mel": 0,   # Melanoma
    "bcc": 1    # Basal Cell Carcinoma
}

ham_cancer = ham[
    ham["dx"].isin(ham_mapping.keys())
].copy()

# ---------------------------------------------------------
# 2. Create standardized dataframe
# ---------------------------------------------------------

ham_df = pd.DataFrame()

ham_df["image_name"] = ham_cancer["image_id"]
ham_df["class_name"] = ham_cancer["dx"].map({
    "mel": "Melanoma",
    "bcc": "Basal Cell Carcinoma"
})
ham_df["label"] = ham_cancer["dx"].map(ham_mapping)
ham_df["age"] = ham_cancer["age"]
ham_df["sex"] = ham_cancer["sex"]
ham_df["dataset"] = "HAM10000"

# ---------------------------------------------------------
# 3. Find actual image path
# ---------------------------------------------------------

HAM_IMAGE_DIRS = [
    os.path.join(HAM_ROOT, "HAM10000_images_part_1"),
    os.path.join(HAM_ROOT, "HAM10000_images_part_2"),
    os.path.join(HAM_ROOT, "ham10000_images_part_1"),
    os.path.join(HAM_ROOT, "ham10000_images_part_2")
]

# Create filename -> full path lookup
ham_image_lookup = {}

for image_dir in HAM_IMAGE_DIRS:

    if os.path.exists(image_dir):

        for filename in os.listdir(image_dir):

            if filename.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):
                image_id = os.path.splitext(filename)[0]
                ham_image_lookup[image_id] = os.path.join(
                    image_dir,
                    filename
                )

ham_df["image_path"] = ham_df["image_name"].map(
    ham_image_lookup
)

# ---------------------------------------------------------
# 4. Validate image paths
# ---------------------------------------------------------

ham_df["image_exists"] = ham_df["image_path"].apply(
    lambda x: os.path.isfile(x)
)

# ---------------------------------------------------------
# 5. Remove records without images
# ---------------------------------------------------------

before_count = len(ham_df)

ham_df = ham_df[
    ham_df["image_exists"]
].copy()

after_count = len(ham_df)

# ---------------------------------------------------------
# 6. Reset index
# ---------------------------------------------------------

ham_df = ham_df.reset_index(drop=True)

# ---------------------------------------------------------
# 7. Save intermediate preprocessing result
# ---------------------------------------------------------

ham_save_path = os.path.join(
    DATA_DIR,
    "ham10000_preprocessed.csv"
)

ham_df.to_csv(
    ham_save_path,
    index=False
)

# ---------------------------------------------------------
# 8. Results
# ---------------------------------------------------------

print("\nHAM10000 relevant records BEFORE image check:", before_count)
print("HAM10000 records AFTER image check :", after_count)
print("Images removed due to missing path :", before_count - after_count)

print("\nClass distribution:")
print(ham_df["class_name"].value_counts())

print("\nLabel distribution:")
print(ham_df["label"].value_counts().sort_index())

print("\nDataset distribution:")
print(ham_df["dataset"].value_counts())

print("\nImage existence:")
print(ham_df["image_exists"].value_counts())

print("\nSample records:")
display(
    ham_df[
        [
            "image_name",
            "class_name",
            "label",
            "age",
            "sex",
            "image_path"
        ]
    ].head()
)

print("\nSaved to:")
print(ham_save_path)

print("\n" + "=" * 70)
print("STEP 4 COMPLETE")
print("=" * 70)

In [ ]:
# STEP 5 — ISIC 2024 Preprocessing

import os
import pandas as pd

print("=" * 70)
print("STEP 5 — ISIC 2024 PREPROCESSING")
print("=" * 70)

# ---------------------------------------------------------
# 1. Define cancer diagnosis mapping
# ---------------------------------------------------------

ISIC_MAPPING = {
    "melanoma": 0,
    "melanoma metastasis": 0,
    "basal cell carcinoma": 1,
    "squamous cell carcinoma": 2
}

ISIC_CLASS_NAMES = {
    0: "Melanoma",
    1: "Basal Cell Carcinoma",
    2: "Squamous Cell Carcinoma"
}

# ---------------------------------------------------------
# 2. Select only required cancer diagnoses
# ---------------------------------------------------------

isic_cancer = isic[
    isic["diagnosis"].astype(str).str.lower().isin(
        ISIC_MAPPING.keys()
    )
].copy()

print("\nSelected ISIC cancer records:", len(isic_cancer))

# ---------------------------------------------------------
# 3. Standardized dataframe
# ---------------------------------------------------------

isic_df = pd.DataFrame()

isic_df["image_name"] = isic_cancer["isic_id"]

isic_df["class_name"] = (
    isic_cancer["diagnosis"]
    .astype(str)
    .str.lower()
    .map(ISIC_MAPPING)
    .map(ISIC_CLASS_NAMES)
)

isic_df["label"] = (
    isic_cancer["diagnosis"]
    .astype(str)
    .str.lower()
    .map(ISIC_MAPPING)
)

isic_df["age"] = isic_cancer["age_approx"]
isic_df["sex"] = isic_cancer["sex"]
isic_df["dataset"] = "ISIC 2024"

# ---------------------------------------------------------
# 4. Build ISIC image lookup
# ---------------------------------------------------------

ISIC_IMAGE_DIR = os.path.join(
    ISIC_ROOT,
    "images"
)

print("\nISIC image directory:")
print(ISIC_IMAGE_DIR)
print("Exists:", os.path.exists(ISIC_IMAGE_DIR))

isic_image_lookup = {}

if os.path.exists(ISIC_IMAGE_DIR):

    for filename in os.listdir(ISIC_IMAGE_DIR):

        if filename.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            image_id = os.path.splitext(filename)[0]

            isic_image_lookup[image_id] = os.path.join(
                ISIC_IMAGE_DIR,
                filename
            )

# ---------------------------------------------------------
# 5. Match metadata IDs to actual images
# ---------------------------------------------------------

isic_df["image_path"] = isic_df["image_name"].map(
    isic_image_lookup
)

# ---------------------------------------------------------
# 6. Check image existence
# ---------------------------------------------------------

isic_df["image_exists"] = isic_df["image_path"].apply(
    lambda x: os.path.isfile(x)
)

missing_count = (~isic_df["image_exists"]).sum()

print("\nMissing image paths:", missing_count)

# ---------------------------------------------------------
# 7. Keep only records with valid images
# ---------------------------------------------------------

before_count = len(isic_df)

isic_df = isic_df[
    isic_df["image_exists"]
].copy()

after_count = len(isic_df)

isic_df = isic_df.reset_index(drop=True)

# ---------------------------------------------------------
# 8. Save intermediate result
# ---------------------------------------------------------

isic_save_path = os.path.join(
    DATA_DIR,
    "isic2024_preprocessed.csv"
)

isic_df.to_csv(
    isic_save_path,
    index=False
)

# ---------------------------------------------------------
# 9. Results
# ---------------------------------------------------------

print("\nRecords BEFORE image check:", before_count)
print("Records AFTER image check :", after_count)
print("Removed records           :", before_count - after_count)

print("\nClass distribution:")
print(isic_df["class_name"].value_counts())

print("\nLabel distribution:")
print(
    isic_df["label"]
    .value_counts()
    .sort_index()
)

print("\nDiagnosis/source distribution:")
print(isic_cancer["diagnosis"].value_counts())

print("\nImage existence:")
print(isic_df["image_exists"].value_counts())

print("\nSample records:")
display(
    isic_df[
        [
            "image_name",
            "class_name",
            "label",
            "age",
            "sex",
            "image_path"
        ]
    ].head()
)

print("\nSaved to:")
print(isic_save_path)

print("\n" + "=" * 70)
print("STEP 5 COMPLETE")
print("=" * 70)

In [ ]:
# STEP 6 — DermaCon-IN Cancer Records Check

import os
import pandas as pd

print("=" * 70)
print("STEP 6 — DERMACON-IN CANCER DATA CHECK")
print("=" * 70)

# ---------------------------------------------------------
# 1. Find all cancer-related disease labels
# ---------------------------------------------------------

cancer_keywords = [
    "melanoma",
    "basal cell",
    "squamous cell",
    "merkel"
]

derma_labels = (
    derma["Disease_label"]
    .astype(str)
    .str.strip()
)

mask = derma_labels.str.lower().str.contains(
    "|".join(cancer_keywords),
    na=False
)

derma_cancer_check = derma[mask].copy()

print("\nCancer-related records found:",
      len(derma_cancer_check))

print("\nCancer-related disease labels:")
print(
    derma_cancer_check["Disease_label"]
    .value_counts()
)

# ---------------------------------------------------------
# 2. Check image directories
# ---------------------------------------------------------

DERMA_IMAGE_ROOT = os.path.join(
    DERMA_ROOT,
    "DATASET"
)

print("\n" + "-" * 70)
print("DermaCon-IN image folders")
print("-" * 70)

for root, dirs, files in os.walk(DERMA_IMAGE_ROOT):

    image_files = [
        f for f in files
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

    if image_files:
        print("\nFolder:", root)
        print("Number of images:", len(image_files))
        print("Sample:", image_files[:5])

# ---------------------------------------------------------
# 3. Build image filename lookup
# ---------------------------------------------------------

derma_image_lookup = {}

for root, dirs, files in os.walk(DERMA_IMAGE_ROOT):

    for filename in files:

        if filename.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            derma_image_lookup[filename] = os.path.join(
                root,
                filename
            )

# ---------------------------------------------------------
# 4. Check whether cancer metadata images exist
# ---------------------------------------------------------

if len(derma_cancer_check) > 0:

    derma_cancer_check["image_path"] = (
        derma_cancer_check["Image_name"]
        .astype(str)
        .map(derma_image_lookup)
    )

    derma_cancer_check["image_exists"] = (
        derma_cancer_check["image_path"]
        .apply(lambda x: os.path.isfile(x))
    )

    print("\n" + "-" * 70)
    print("Cancer metadata → image matching")
    print("-" * 70)

    print(
        derma_cancer_check["image_exists"]
        .value_counts()
    )

    print("\nSample matched records:")

    display(
        derma_cancer_check[
            [
                "Image_name",
                "Disease_label",
                "Age",
                "Sex",
                "image_path",
                "image_exists"
            ]
        ].head(10)
    )

else:

    print("\nNo cancer-related DermaCon-IN records found.")

print("\n" + "=" * 70)
print("STEP 6 COMPLETE")
print("=" * 70)

In [ ]:
# STEP 7 — Dataset Overlap & Duplicate Check

import os
import pandas as pd
from collections import Counter

print("=" * 70)
print("STEP 7 — DATASET OVERLAP & DUPLICATE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load the two preprocessed datasets created earlier
# ------------------------------------------------------------

ham_csv = os.path.join(DATA_DIR, "ham10000_preprocessed.csv")
isic_csv = os.path.join(DATA_DIR, "isic2024_preprocessed.csv")

ham_df = pd.read_csv(ham_csv)
isic_df = pd.read_csv(isic_csv)

print("\nLoaded datasets:")
print(f"HAM10000 : {len(ham_df)}")
print(f"ISIC     : {len(isic_df)}")

# ------------------------------------------------------------
# 2. Check filename overlap
# ------------------------------------------------------------

ham_names = set(
    ham_df["image_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

isic_names = set(
    isic_df["image_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

name_overlap = ham_names.intersection(isic_names)

print("\n" + "-" * 70)
print("HAM10000 ↔ ISIC filename overlap")
print("-" * 70)

print(f"HAM unique filenames : {len(ham_names)}")
print(f"ISIC unique filenames: {len(isic_names)}")
print(f"Overlapping filenames: {len(name_overlap)}")

if len(name_overlap) > 0:
    print("\nSample overlapping filenames:")
    for x in sorted(list(name_overlap))[:20]:
        print(x)

# ------------------------------------------------------------
# 3. DermaCon cancer records
# ------------------------------------------------------------

derma_cancer = derma_cancer_check.copy()

print("\n" + "-" * 70)
print("DermaCon-IN cancer records")
print("-" * 70)

print(f"DermaCon cancer records: {len(derma_cancer)}")

print("\nClass distribution:")
print(
    derma_cancer["Disease_label"]
    .value_counts()
)

# ------------------------------------------------------------
# 4. Compare DermaCon filenames with HAM and ISIC
# ------------------------------------------------------------

derma_names = set(
    derma_cancer["Image_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

derma_ham_overlap = derma_names.intersection(ham_names)
derma_isic_overlap = derma_names.intersection(isic_names)

print("\n" + "-" * 70)
print("DermaCon filename overlap")
print("-" * 70)

print(f"DermaCon ↔ HAM  : {len(derma_ham_overlap)}")
print(f"DermaCon ↔ ISIC : {len(derma_isic_overlap)}")

# ------------------------------------------------------------
# 5. Final decision information
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 7 COMPLETE")
print("=" * 70)

print("\nNext step:")
print("Based on these overlap counts, we will decide which datasets")
print("should enter the FINAL classification dataset.")

In [ ]:
# STEP 8 — Find Candidate SCC Dataset Sources

import os

print("=" * 70)
print("STEP 8 — CANDIDATE SCC DATASET SOURCES")
print("=" * 70)

DATASETS_ROOT = "/kaggle/input/datasets"

# Known dataset roots from our project
candidate_roots = {
    "ahmedxc4_skin_ds":
        os.path.join(DATASETS_ROOT, "ahmedxc4", "skin-ds"),

    "riyaelizashaju_isic_labelled":
        os.path.join(
            DATASETS_ROOT,
            "riyaelizashaju",
            "isic-skin-disease-image-dataset-labelled"
        ),

    "nodoubttome_skin_cancer9":
        os.path.join(
            DATASETS_ROOT,
            "nodoubttome",
            "skin-cancer9-classesisic"
        ),
}

for name, path in candidate_roots.items():

    print("\n" + "-" * 70)
    print(name)
    print("-" * 70)

    if not os.path.exists(path):
        print("Path NOT FOUND:")
        print(path)
        continue

    print("Path FOUND:")
    print(path)

    # Show immediate folders/files
    try:
        items = sorted(os.listdir(path))
        print("\nContents:")
        for item in items[:50]:
            full_path = os.path.join(path, item)

            if os.path.isdir(full_path):
                print(f"[DIR ] {item}")
            else:
                print(f"[FILE] {item}")

        if len(items) > 50:
            print(f"... and {len(items) - 50} more items")

    except Exception as e:
        print("Could not inspect:", e)

print("\n" + "=" * 70)
print("STEP 8 COMPLETE")
print("=" * 70)

In [ ]:
# STEP 9 — Count SCC Images in Candidate Datasets

import os

print("=" * 70)
print("STEP 9 — SCC IMAGE COUNT & FOLDER INSPECTION")
print("=" * 70)

candidate_roots = {
    "AhmedXC4":
        "/kaggle/input/datasets/ahmedxc4/skin-ds",

    "RiyaElizashaju":
        "/kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled",

    "NoDoubtToMe":
        "/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic"
}

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

for dataset_name, root in candidate_roots.items():

    print("\n" + "-" * 70)
    print(dataset_name)
    print("-" * 70)

    if not os.path.exists(root):
        print("NOT FOUND:", root)
        continue

    scc_files = []

    # Search all folders/files recursively
    for current_root, dirs, files in os.walk(root):

        folder_name = os.path.basename(current_root).lower()

        # Only inspect folders whose name contains SCC / squamous
        if (
            "squamous" in folder_name
            or folder_name == "scc"
            or "scc" in folder_name
        ):

            for file_name in files:

                ext = os.path.splitext(file_name)[1].lower()

                if ext in IMAGE_EXTENSIONS:
                    scc_files.append(
                        os.path.join(current_root, file_name)
                    )

    print("SCC image count:", len(scc_files))

    # Show folders containing SCC images
    folders = sorted(
        set(os.path.dirname(x) for x in scc_files)
    )

    print("\nSCC folders found:")

    if folders:
        for folder in folders:
            print(folder)
    else:
        print("No SCC-labelled folder detected.")

    print("\nSample SCC filenames:")

    for path in scc_files[:20]:
        print(os.path.basename(path))

print("\n" + "=" * 70)
print("STEP 9 COMPLETE")
print("=" * 70)

In [ ]:
# STEP 10 — Exact Unique SCC Images & ISIC Overlap

import os
import pandas as pd

print("=" * 70)
print("STEP 10 — EXACT UNIQUE SCC IMAGES & ISIC OVERLAP")
print("=" * 70)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

scc_roots = {
    "AhmedXC4":
        "/kaggle/input/datasets/ahmedxc4/skin-ds",

    "RiyaElizashaju":
        "/kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled",

    "NoDoubtToMe":
        "/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic"
}

# ------------------------------------------------------------
# 1. Collect SCC files using filenames
# ------------------------------------------------------------

dataset_files = {}

for dataset_name, root in scc_roots.items():

    files_found = []

    for current_root, dirs, files in os.walk(root):

        folder_name = os.path.basename(current_root).lower()

        if (
            "squamous cell carcinoma" in folder_name
            or folder_name == "scc"
        ):

            for file_name in files:

                ext = os.path.splitext(file_name)[1].lower()

                if ext in IMAGE_EXTENSIONS:
                    files_found.append(
                        os.path.join(current_root, file_name)
                    )

    dataset_files[dataset_name] = files_found

# ------------------------------------------------------------
# 2. Convert filenames to normalized IDs
# ------------------------------------------------------------

dataset_ids = {}

for dataset_name, files_found in dataset_files.items():

    ids = set()

    for path in files_found:

        file_name = os.path.basename(path)

        # Remove extension
        image_id = os.path.splitext(file_name)[0].lower().strip()

        ids.add(image_id)

    dataset_ids[dataset_name] = ids

# ------------------------------------------------------------
# 3. Print counts
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SCC COUNTS")
print("-" * 70)

for name, ids in dataset_ids.items():
    print(f"{name:20s}: {len(ids)} unique filenames")

# ------------------------------------------------------------
# 4. Pairwise overlaps between SCC datasets
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("PAIRWISE SCC OVERLAPS")
print("-" * 70)

names = list(dataset_ids.keys())

for i in range(len(names)):
    for j in range(i + 1, len(names)):

        a = names[i]
        b = names[j]

        overlap = dataset_ids[a].intersection(dataset_ids[b])

        print(f"{a} ↔ {b}: {len(overlap)}")

        if overlap:
            print("  Sample:", sorted(list(overlap))[:10])

# ------------------------------------------------------------
# 5. Combined unique SCC set
# ------------------------------------------------------------

all_scc_ids = set().union(*dataset_ids.values())

print("\n" + "-" * 70)
print("COMBINED SCC SET")
print("-" * 70)

print("Total unique SCC IDs:", len(all_scc_ids))

# ------------------------------------------------------------
# 6. Compare against ISIC 2024
# ------------------------------------------------------------

isic_ids = set(
    isic_df["image_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

scc_isic_overlap = all_scc_ids.intersection(isic_ids)

print("\n" + "-" * 70)
print("SCC ↔ ISIC OVERLAP")
print("-" * 70)

print("ISIC unique IDs:", len(isic_ids))
print("Unique SCC IDs :", len(all_scc_ids))
print("SCC IDs already in ISIC:", len(scc_isic_overlap))
print(
    "Potentially new SCC IDs:",
    len(all_scc_ids - isic_ids)
)

if scc_isic_overlap:
    print("\nSample SCC IDs already present in ISIC:")
    for x in sorted(list(scc_isic_overlap))[:20]:
        print(x)

# ------------------------------------------------------------
# 7. Check overlap with DermaCon
# ------------------------------------------------------------

derma_ids = set(
    derma_cancer["Image_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

scc_derma_overlap = all_scc_ids.intersection(derma_ids)

print("\n" + "-" * 70)
print("SCC ↔ DERMACON OVERLAP")
print("-" * 70)

print("SCC ↔ DermaCon overlap:", len(scc_derma_overlap))

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 10 COMPLETE")
print("=" * 70)

print("\nThese numbers will be used to build the final")
print("duplicate-safe classification dataset.")

In [ ]:
# STEP 11 — Build Duplicate-Safe Final 4-Class Dataset

import os
import pandas as pd

print("=" * 70)
print("STEP 11 — FINAL 4-CLASS DATASET BUILD")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load ISIC preprocessed data
# ------------------------------------------------------------

isic_final = isic_df.copy()

# Standardize columns
isic_final = isic_final[
    [
        "image_name",
        "class_name",
        "label",
        "age",
        "sex",
        "dataset",
        "image_path"
    ]
].copy()

print("\nISIC records:", len(isic_final))

# ------------------------------------------------------------
# 2. Prepare DermaCon cancer records
# ------------------------------------------------------------

derma_final = derma_cancer.copy()

derma_label_map = {
    "Melanoma": 0,
    "Basal Cell Carcinoma": 1,
    "Squamous Cell Carcinoma": 2
}

derma_final["class_name"] = (
    derma_final["Disease_label"]
    .map(derma_label_map)
    .map({
        0: "Melanoma",
        1: "BCC",
        2: "SCC"
    })
)

derma_final["label"] = (
    derma_final["Disease_label"]
    .map(derma_label_map)
)

derma_final["image_name"] = (
    derma_final["Image_name"]
    .astype(str)
)

derma_final["age"] = derma_final["Age"].astype(str)
derma_final["sex"] = derma_final["Sex"].astype(str)
derma_final["dataset"] = "DermaCon-IN"

derma_final = derma_final[
    [
        "image_name",
        "class_name",
        "label",
        "age",
        "sex",
        "dataset",
        "image_path"
    ]
].copy()

print("DermaCon cancer records:", len(derma_final))

# ------------------------------------------------------------
# 3. Prepare existing MCC dataset
# ------------------------------------------------------------

MCC_EXISTING_ROOT = (
    "/kaggle/input/datasets/quantumcoders03/datasetmcc"
)

mcc_existing_files = []

for current_root, dirs, files in os.walk(MCC_EXISTING_ROOT):

    for file_name in files:

        ext = os.path.splitext(file_name)[1].lower()

        if ext in IMAGE_EXTENSIONS:

            mcc_existing_files.append(
                os.path.join(current_root, file_name)
            )

mcc_existing_df = pd.DataFrame({
    "image_name": [
        os.path.splitext(os.path.basename(x))[0]
        for x in mcc_existing_files
    ],
    "class_name": "MCC",
    "label": 3,
    "age": "unknown",
    "sex": "unknown",
    "dataset": "MCC-existing",
    "image_path": mcc_existing_files
})

print("Existing MCC images:", len(mcc_existing_df))

# ------------------------------------------------------------
# 4. Prepare new MCC datasets
# ------------------------------------------------------------

new_mcc_files = []

for root in MCC_NEW_ROOTS:

    if not os.path.exists(root):
        print("WARNING — MCC path not found:", root)
        continue

    for current_root, dirs, files in os.walk(root):

        for file_name in files:

            ext = os.path.splitext(file_name)[1].lower()

            if ext in IMAGE_EXTENSIONS:

                new_mcc_files.append(
                    os.path.join(current_root, file_name)
                )

print("Raw new MCC image files:", len(new_mcc_files))

new_mcc_df = pd.DataFrame({
    "image_name": [
        os.path.splitext(os.path.basename(x))[0]
        for x in new_mcc_files
    ],
    "class_name": "MCC",
    "label": 3,
    "age": "unknown",
    "sex": "unknown",
    "dataset": "MCC-new",
    "image_path": new_mcc_files
})

# ------------------------------------------------------------
# 5. Remove duplicate MCC filenames
# ------------------------------------------------------------

mcc_existing_ids = set(
    mcc_existing_df["image_name"]
    .astype(str)
    .str.lower()
)

new_mcc_df["normalized_id"] = (
    new_mcc_df["image_name"]
    .astype(str)
    .str.lower()
)

# Remove duplicates inside new MCC datasets
new_mcc_df = new_mcc_df.drop_duplicates(
    subset=["normalized_id"]
).copy()

# Remove images already present in existing MCC
new_mcc_df = new_mcc_df[
    ~new_mcc_df["normalized_id"].isin(mcc_existing_ids)
].copy()

new_mcc_df = new_mcc_df.drop(
    columns=["normalized_id"]
)

print("Unique new MCC images:", len(new_mcc_df))

# ------------------------------------------------------------
# 6. Combine all valid sources
# ------------------------------------------------------------

final_df = pd.concat(
    [
        isic_final,
        derma_final,
        mcc_existing_df,
        new_mcc_df
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 7. Final duplicate protection
# ------------------------------------------------------------

final_df["normalized_id"] = (
    final_df["image_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

duplicate_count = (
    final_df["normalized_id"]
    .duplicated()
    .sum()
)

print("\nDuplicate IDs before final removal:", duplicate_count)

final_df = final_df.drop_duplicates(
    subset=["normalized_id"],
    keep="first"
).copy()

final_df = final_df.drop(
    columns=["normalized_id"]
)

# ------------------------------------------------------------
# 8. Verify image paths
# ------------------------------------------------------------

final_df["image_exists"] = (
    final_df["image_path"]
    .apply(os.path.isfile)
)

missing_images = (
    (~final_df["image_exists"])
    .sum()
)

print("Missing image files:", missing_images)

final_df = final_df[
    final_df["image_exists"]
].copy()

final_df = final_df.drop(
    columns=["image_exists"]
)

# ------------------------------------------------------------
# 9. Final class distribution
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FINAL CLASS DISTRIBUTION")
print("-" * 70)

print(
    final_df["class_name"]
    .value_counts()
    .reindex(CLASS_NAMES)
)

print("\nTotal final images:", len(final_df))

# ------------------------------------------------------------
# 10. Dataset-source distribution
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("DATASET SOURCE DISTRIBUTION")
print("-" * 70)

print(
    final_df["dataset"]
    .value_counts()
)

# ------------------------------------------------------------
# 11. Save final classification CSV
# ------------------------------------------------------------

FINAL_CLASSIFICATION_CSV = os.path.join(
    DATA_DIR,
    "final_4class_classification_dataset.csv"
)

final_df.to_csv(
    FINAL_CLASSIFICATION_CSV,
    index=False
)

print("\nSaved:")
print(FINAL_CLASSIFICATION_CSV)

print("\n" + "=" * 70)
print("STEP 11 COMPLETE")
print("=" * 70)

In [ ]:
# STEP 12 — Fix Class Labels + Content-Based Duplicate Check

import os
import hashlib
import pandas as pd

print("=" * 70)
print("STEP 12 — CLASS LABEL FIX + CONTENT DUPLICATE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Reload source datasets
# ------------------------------------------------------------

isic_clean = isic_df.copy()
derma_clean = derma_cancer.copy()

# ------------------------------------------------------------
# 2. Standardize ISIC class names
# ------------------------------------------------------------

isic_class_map = {
    0: "Melanoma",
    1: "BCC",
    2: "SCC"
}

isic_clean["class_name"] = (
    isic_clean["label"].map(isic_class_map)
)

isic_clean["dataset"] = "ISIC 2024"

isic_clean = isic_clean[
    [
        "image_name",
        "class_name",
        "label",
        "age",
        "sex",
        "dataset",
        "image_path"
    ]
].copy()

# ------------------------------------------------------------
# 3. Standardize DermaCon class names
# ------------------------------------------------------------

derma_class_map = {
    "Melanoma": ("Melanoma", 0),
    "Basal Cell Carcinoma": ("BCC", 1),
    "Squamous Cell Carcinoma": ("SCC", 2)
}

derma_clean["class_name"] = (
    derma_clean["Disease_label"]
    .map(lambda x: derma_class_map[x][0])
)

derma_clean["label"] = (
    derma_clean["Disease_label"]
    .map(lambda x: derma_class_map[x][1])
)

derma_clean["image_name"] = (
    derma_clean["Image_name"].astype(str)
)

derma_clean["age"] = derma_clean["Age"].astype(str)
derma_clean["sex"] = derma_clean["Sex"].astype(str)
derma_clean["dataset"] = "DermaCon-IN"

derma_clean = derma_clean[
    [
        "image_name",
        "class_name",
        "label",
        "age",
        "sex",
        "dataset",
        "image_path"
    ]
].copy()

# ------------------------------------------------------------
# 4. Collect existing MCC images
# ------------------------------------------------------------

mcc_existing_files = []

for current_root, dirs, files in os.walk(MCC_EXISTING_ROOT):

    for file_name in files:

        ext = os.path.splitext(file_name)[1].lower()

        if ext in IMAGE_EXTENSIONS:
            mcc_existing_files.append(
                os.path.join(current_root, file_name)
            )

mcc_existing_clean = pd.DataFrame({
    "image_name": [
        os.path.splitext(os.path.basename(x))[0]
        for x in mcc_existing_files
    ],
    "class_name": "MCC",
    "label": 3,
    "age": "unknown",
    "sex": "unknown",
    "dataset": "MCC-existing",
    "image_path": mcc_existing_files
})

# ------------------------------------------------------------
# 5. Collect all new MCC images
# ------------------------------------------------------------

new_mcc_files = []

for root in MCC_NEW_ROOTS:

    if not os.path.exists(root):
        continue

    for current_root, dirs, files in os.walk(root):

        for file_name in files:

            ext = os.path.splitext(file_name)[1].lower()

            if ext in IMAGE_EXTENSIONS:
                new_mcc_files.append(
                    os.path.join(current_root, file_name)
                )

mcc_new_clean = pd.DataFrame({
    "image_name": [
        os.path.splitext(os.path.basename(x))[0]
        for x in new_mcc_files
    ],
    "class_name": "MCC",
    "label": 3,
    "age": "unknown",
    "sex": "unknown",
    "dataset": "MCC-new",
    "image_path": new_mcc_files
})

# ------------------------------------------------------------
# 6. Calculate SHA256 image hashes
# ------------------------------------------------------------

def file_hash(path):
    sha = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(chunk)

    return sha.hexdigest()

print("\nCalculating image hashes...")

all_mcc = pd.concat(
    [
        mcc_existing_clean,
        mcc_new_clean
    ],
    ignore_index=True
)

all_mcc["image_hash"] = (
    all_mcc["image_path"].apply(file_hash)
)

# ------------------------------------------------------------
# 7. Detect exact duplicate MCC images
# ------------------------------------------------------------

duplicate_hashes = (
    all_mcc["image_hash"]
    .duplicated(keep=False)
)

mcc_duplicates = all_mcc[
    duplicate_hashes
].sort_values("image_hash")

print("\n" + "-" * 70)
print("MCC CONTENT DUPLICATE CHECK")
print("-" * 70)

print(
    "Total MCC files:",
    len(all_mcc)
)

print(
    "Unique MCC image contents:",
    all_mcc["image_hash"].nunique()
)

print(
    "Duplicate MCC files:",
    duplicate_hashes.sum()
)

print(
    "Duplicate image groups:",
    mcc_duplicates["image_hash"].nunique()
)

if len(mcc_duplicates) > 0:

    print("\nDuplicate groups:")

    for h, group in mcc_duplicates.groupby("image_hash"):

        print("\nHash:", h[:16] + "...")

        for _, row in group.iterrows():
            print(
                f"  {row['dataset']:15s} | "
                f"{row['image_name']}"
            )

# ------------------------------------------------------------
# 8. Keep only one copy of each MCC image
# ------------------------------------------------------------

mcc_final = (
    all_mcc
    .drop_duplicates(
        subset=["image_hash"],
        keep="first"
    )
    .drop(columns=["image_hash"])
    .copy()
)

# ------------------------------------------------------------
# 9. Combine ISIC + DermaCon + MCC
# ------------------------------------------------------------

final_fixed = pd.concat(
    [
        isic_clean,
        derma_clean,
        mcc_final
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 10. Content-based duplicate check across ALL sources
# ------------------------------------------------------------

print("\nCalculating final dataset image hashes...")

final_fixed["image_hash"] = (
    final_fixed["image_path"].apply(file_hash)
)

final_duplicate_count = (
    final_fixed["image_hash"]
    .duplicated()
    .sum()
)

print("\n" + "-" * 70)
print("FINAL CROSS-SOURCE CONTENT DUPLICATE CHECK")
print("-" * 70)

print(
    "Total records before final deduplication:",
    len(final_fixed)
)

print(
    "Duplicate image contents:",
    final_duplicate_count
)

final_fixed = (
    final_fixed
    .drop_duplicates(
        subset=["image_hash"],
        keep="first"
    )
    .copy()
)

final_fixed = final_fixed.drop(
    columns=["image_hash"]
)

# ------------------------------------------------------------
# 11. Verify all image files
# ------------------------------------------------------------

final_fixed["image_exists"] = (
    final_fixed["image_path"].apply(os.path.isfile)
)

missing_count = (
    (~final_fixed["image_exists"]).sum()
)

final_fixed = final_fixed[
    final_fixed["image_exists"]
].drop(columns=["image_exists"])

# ------------------------------------------------------------
# 12. Final class distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL CORRECTED CLASS DISTRIBUTION")
print("=" * 70)

class_counts = (
    final_fixed["class_name"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)

print(class_counts)

print("\nTotal final images:", len(final_fixed))

# ------------------------------------------------------------
# 13. Dataset source distribution
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FINAL DATASET SOURCE DISTRIBUTION")
print("-" * 70)

print(
    final_fixed["dataset"].value_counts()
)

# ------------------------------------------------------------
# 14. Save corrected dataset
# ------------------------------------------------------------

FINAL_CLASSIFICATION_CSV = os.path.join(
    DATA_DIR,
    "final_4class_classification_dataset.csv"
)

final_fixed.to_csv(
    FINAL_CLASSIFICATION_CSV,
    index=False
)

print("\nSaved corrected dataset:")
print(FINAL_CLASSIFICATION_CSV)

print("\nMissing images:", missing_count)

print("\n" + "=" * 70)
print("STEP 12 COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# MCC DATASET CHECK
# ================================================================

import os

print("=" * 70)
print("SEARCHING FOR MCC DATASETS")
print("=" * 70)

mcc_keywords = [
    "mcc",
    "merkel",
    "merkel cell"
]

found = []

for root, dirs, files in os.walk("/kaggle/input/datasets"):

    root_lower = root.lower()

    if any(k in root_lower for k in mcc_keywords):

        if root not in found:
            found.append(root)

for root in sorted(found):
    print(root)

print("\n" + "-" * 70)
print("MCC-RELATED DIRECTORIES:", len(found))
print("-" * 70)

if len(found) == 0:
    print("NO MCC DATASET FOUND.")
    print("\nPlease attach the MCC dataset using:")
    print("Right side → Input → Add Input → Datasets")

print("\n" + "=" * 70)

In [ ]:
# ================================================================
# STEP 12A — MCC DATASET AUDIT
# ================================================================

import os
import hashlib
import pandas as pd

MCC_ROOTS = [
    "/kaggle/input/datasets/quantumcoders07/datasetmcc",
    "/kaggle/input/datasets/quantumcoders07/mcc-1-dataset",
    "/kaggle/input/datasets/quantumcoders07/mcc-2-datatset",
    "/kaggle/input/datasets/quantumcoders07/mcc-3-datatset",
]

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

records = []

for root in MCC_ROOTS:

    if not os.path.exists(root):
        print("NOT FOUND:", root)
        continue

    for current_root, dirs, files in os.walk(root):

        for file in files:

            ext = os.path.splitext(file)[1].lower()

            if ext in IMAGE_EXTENSIONS:

                full_path = os.path.join(current_root, file)

                records.append({
                    "source_root": root,
                    "image_path": full_path,
                    "filename": file
                })

mcc_df = pd.DataFrame(records)

print("=" * 70)
print("MCC DATASET AUDIT")
print("=" * 70)

print("\nTotal MCC candidate images:", len(mcc_df))

print("\nImages by source:")
print(
    mcc_df["source_root"]
    .value_counts()
)

# ------------------------------------------------
# SHA256 duplicate check
# ------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

print("\nCalculating SHA-256 hashes...")

mcc_df["sha256"] = mcc_df["image_path"].apply(sha256_file)

duplicate_mask = mcc_df.duplicated(
    subset="sha256",
    keep=False
)

duplicates = mcc_df[duplicate_mask].copy()

print("\nUnique MCC images:",
      mcc_df["sha256"].nunique())

print("Duplicate image records:",
      len(duplicates))

print("Duplicate groups:",
      duplicates["sha256"].nunique())

print("\n" + "-" * 70)
print("DUPLICATE GROUPS")
print("-" * 70)

if len(duplicates) > 0:

    for sha, group in duplicates.groupby("sha256"):

        print("\nSHA256:", sha)

        for path in group["image_path"]:
            print("  ", path)

else:
    print("No exact duplicates found.")

# ------------------------------------------------
# Save audit
# ------------------------------------------------

audit_path = (
    "/kaggle/working/skin_cancer_project/data/"
    "mcc_dataset_audit.csv"
)

os.makedirs(
    os.path.dirname(audit_path),
    exist_ok=True
)

mcc_df.to_csv(
    audit_path,
    index=False
)

print("\nSaved:")
print(audit_path)

print("\n" + "=" * 70)
print("MCC DATASET AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# STEP 12B — BUILD UNIQUE MCC DATASET
# ================================================================

import os
import shutil
import pandas as pd

MCC_OUTPUT_DIR = (
    "/kaggle/working/skin_cancer_project/data/"
    "mcc_unique"
)

os.makedirs(MCC_OUTPUT_DIR, exist_ok=True)

# mcc_df came from STEP 12A
# Keep exactly one image from each SHA-256 group
unique_mcc_df = (
    mcc_df
    .drop_duplicates(subset="sha256", keep="first")
    .copy()
    .reset_index(drop=True)
)

print("=" * 70)
print("BUILDING UNIQUE MCC DATASET")
print("=" * 70)

print("\nCandidate images :", len(mcc_df))
print("Unique MCC images:", len(unique_mcc_df))
print("Removed duplicates:",
      len(mcc_df) - len(unique_mcc_df))

# ------------------------------------------------
# Copy unique MCC images
# ------------------------------------------------

output_records = []

for i, row in unique_mcc_df.iterrows():

    src = row["image_path"]

    ext = os.path.splitext(src)[1].lower()

    new_name = f"MCC_{i+1:04d}{ext}"

    dst = os.path.join(
        MCC_OUTPUT_DIR,
        new_name
    )

    shutil.copy2(src, dst)

    output_records.append({
        "image_name": new_name,
        "image_path": dst,
        "class_name": "MCC",
        "label": 3,
        "source_path": src,
        "sha256": row["sha256"]
    })

unique_mcc_manifest = pd.DataFrame(
    output_records
)

manifest_path = (
    "/kaggle/working/skin_cancer_project/data/"
    "unique_mcc_manifest.csv"
)

unique_mcc_manifest.to_csv(
    manifest_path,
    index=False
)

print("\nOutput directory:")
print(MCC_OUTPUT_DIR)

print("\nManifest:")
print(manifest_path)

print("\nFinal unique MCC images:",
      len(unique_mcc_manifest))

print("\nClass:")
print(unique_mcc_manifest["class_name"].value_counts())

print("\n" + "=" * 70)
print("STEP 12B COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# STEP 12C — MCC vs EXISTING DATASET OVERLAP AUDIT
# ================================================================

import os
import hashlib
import pandas as pd

print("=" * 70)
print("STEP 12C — MCC vs EXISTING DATASET OVERLAP AUDIT")
print("=" * 70)

# ------------------------------------------------
# Existing dataset manifests
# ------------------------------------------------

manifest_paths = [
    "/kaggle/working/skin_cancer_project/data/train.csv",
    "/kaggle/working/skin_cancer_project/data/val.csv",
    "/kaggle/working/skin_cancer_project/data/test.csv"
]

existing_records = []

for manifest_path in manifest_paths:

    if not os.path.exists(manifest_path):
        print("WARNING - Missing:", manifest_path)
        continue

    df = pd.read_csv(manifest_path)

    split_name = os.path.basename(
        manifest_path
    ).replace(".csv", "")

    print(
        f"\n{split_name.upper():<12}: "
        f"{len(df)} rows"
    )

    for _, row in df.iterrows():

        image_path = row["image_path"]

        if os.path.exists(image_path):

            existing_records.append({
                "split": split_name,
                "image_path": image_path,
                "class_name": row["class_name"]
            })

print("\nExisting dataset image files found:",
      len(existing_records))

# ------------------------------------------------
# SHA-256 function
# ------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

# ------------------------------------------------
# Hash existing dataset
# ------------------------------------------------

print("\nHashing existing dataset...")

existing_df = pd.DataFrame(existing_records)

existing_df["sha256"] = (
    existing_df["image_path"]
    .apply(sha256_file)
)

# ------------------------------------------------
# Hash unique MCC dataset
# ------------------------------------------------

print("Hashing unique MCC dataset...")

mcc_unique_df = pd.read_csv(
    "/kaggle/working/skin_cancer_project/data/"
    "unique_mcc_manifest.csv"
)

mcc_unique_df["sha256"] = (
    mcc_unique_df["image_path"]
    .apply(sha256_file)
)

# ------------------------------------------------
# Find exact overlaps
# ------------------------------------------------

existing_hash_map = {}

for _, row in existing_df.iterrows():

    existing_hash_map.setdefault(
        row["sha256"],
        []
    ).append({
        "split": row["split"],
        "image_path": row["image_path"],
        "class_name": row["class_name"]
    })

overlap_records = []

for _, row in mcc_unique_df.iterrows():

    sha = row["sha256"]

    if sha in existing_hash_map:

        for match in existing_hash_map[sha]:

            overlap_records.append({
                "mcc_image": row["image_path"],
                "mcc_source": row["source_path"],
                "split": match["split"],
                "existing_image": match["image_path"],
                "existing_class": match["class_name"],
                "sha256": sha
            })

overlap_df = pd.DataFrame(
    overlap_records
)

# ------------------------------------------------
# Results
# ------------------------------------------------

print("\n" + "-" * 70)
print("OVERLAP RESULTS")
print("-" * 70)

print(
    "\nUnique MCC images checked:",
    len(mcc_unique_df)
)

print(
    "Exact overlaps found:",
    len(overlap_df)
)

if len(overlap_df) > 0:

    print(
        "\nMCC images overlapping existing dataset:",
        overlap_df["mcc_image"].nunique()
    )

    print("\nOverlap by split:")
    print(
        overlap_df["split"].value_counts()
    )

    print("\nOVERLAPPING FILES:")
    display(overlap_df)

else:

    print(
        "\nPASS — No exact SHA-256 overlap found."
    )

# ------------------------------------------------
# Save audit
# ------------------------------------------------

audit_path = (
    "/kaggle/working/skin_cancer_project/data/"
    "step12c_mcc_existing_overlap_audit.csv"
)

overlap_df.to_csv(
    audit_path,
    index=False
)

print("\nSaved:")
print(audit_path)

print("\n" + "=" * 70)
print("STEP 12C COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# STEP 12D — MERGE UNIQUE MCC INTO FINAL 4-CLASS DATASET
# ================================================================

import os
import pandas as pd

print("=" * 70)
print("STEP 12D — MERGING MCC INTO FINAL DATASET")
print("=" * 70)

# ------------------------------------------------
# Existing master dataset
# ------------------------------------------------

MASTER_PATH = (
    "/kaggle/working/skin_cancer_project/data/"
    "final_4class_classification_dataset.csv"
)

MCC_MANIFEST_PATH = (
    "/kaggle/working/skin_cancer_project/data/"
    "unique_mcc_manifest.csv"
)

if not os.path.exists(MASTER_PATH):
    raise FileNotFoundError(
        f"Master dataset not found:\n{MASTER_PATH}"
    )

if not os.path.exists(MCC_MANIFEST_PATH):
    raise FileNotFoundError(
        f"MCC manifest not found:\n{MCC_MANIFEST_PATH}"
    )

# ------------------------------------------------
# Load datasets
# ------------------------------------------------

master_df = pd.read_csv(MASTER_PATH)
mcc_df = pd.read_csv(MCC_MANIFEST_PATH)

print("\nExisting master dataset:", len(master_df))
print("Unique MCC images:", len(mcc_df))

# ------------------------------------------------
# Standardize MCC records
# ------------------------------------------------

mcc_add = pd.DataFrame({
    "image_name": mcc_df["image_name"],
    "class_name": "MCC",
    "label": 3,
    "image_path": mcc_df["image_path"]
})

# Keep any extra columns from master dataset
for col in master_df.columns:

    if col not in mcc_add.columns:

        mcc_add[col] = pd.NA

# Same column order
mcc_add = mcc_add[
    master_df.columns
]

# ------------------------------------------------
# Merge
# ------------------------------------------------

final_df = pd.concat(
    [master_df, mcc_add],
    ignore_index=True
)

# Remove accidental exact image-path duplicates
final_df = final_df.drop_duplicates(
    subset=["image_path"],
    keep="first"
).reset_index(drop=True)

# ------------------------------------------------
# Verify
# ------------------------------------------------

print("\n" + "-" * 70)
print("FINAL DATASET DISTRIBUTION")
print("-" * 70)

print(
    final_df["class_name"]
    .value_counts()
)

print("\nTotal images:", len(final_df))

expected_total = len(master_df) + len(mcc_df)

print("Expected total:", expected_total)

print(
    "Total correct:",
    len(final_df) == expected_total
)

# ------------------------------------------------
# Save
# ------------------------------------------------

FINAL_PATH = (
    "/kaggle/working/skin_cancer_project/data/"
    "final_4class_classification_dataset_with_mcc.csv"
)

final_df.to_csv(
    FINAL_PATH,
    index=False
)

print("\nSaved:")
print(FINAL_PATH)

print("\n" + "=" * 70)
print("STEP 12D COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# STEP 12E — RANDOM VISUAL QUALITY CHECK
# ================================================================

import os
import random
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

print("=" * 70)
print("STEP 12E — RANDOM VISUAL QUALITY CHECK")
print("=" * 70)

# ------------------------------------------------
# 1. Load FINAL dataset INCLUDING MCC
# ------------------------------------------------

FINAL_CLASSIFICATION_CSV = os.path.join(
    DATA_DIR,
    "final_4class_classification_dataset_with_mcc.csv"
)

visual_df = pd.read_csv(FINAL_CLASSIFICATION_CSV)

print("Total images:", len(visual_df))

# ------------------------------------------------
# 2. Check required columns
# ------------------------------------------------

required_columns = [
    "image_path",
    "class_name",
    "label"
]

missing_columns = [
    col for col in required_columns
    if col not in visual_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------
# 3. Check class distribution
# ------------------------------------------------

print("\nClass distribution:")
print(
    visual_df["class_name"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)

# ------------------------------------------------
# 4. Random sample from EACH class
# ------------------------------------------------

SAMPLES_PER_CLASS = 4

random.seed(SEED)

sample_rows = []

for class_name in CLASS_NAMES:

    class_df = visual_df[
        visual_df["class_name"] == class_name
    ]

    if len(class_df) == 0:
        print(
            f"WARNING: No images found for {class_name}"
        )
        continue

    n_samples = min(
        SAMPLES_PER_CLASS,
        len(class_df)
    )

    sampled = class_df.sample(
        n=n_samples,
        random_state=SEED
    )

    sample_rows.append(sampled)

visual_samples = pd.concat(
    sample_rows,
    ignore_index=True
)

print("\nRandom images selected:", len(visual_samples))

# ------------------------------------------------
# 5. Create 4 × 4 image grid
# ------------------------------------------------

fig, axes = plt.subplots(
    4,
    4,
    figsize=(16, 16)
)

axes = axes.flatten()

for i, (_, row) in enumerate(
    visual_samples.iterrows()
):

    ax = axes[i]

    image_path = row["image_path"]
    class_name = row["class_name"]

    # --------------------------------------------
    # Try opening image
    # --------------------------------------------

    try:

        img = Image.open(image_path)

        # Original resolution
        width, height = img.size

        ax.imshow(img)

        ax.set_title(
            f"{class_name}\n{width}×{height}",
            fontsize=10
        )

        ax.axis("off")

    except Exception as e:

        ax.text(
            0.5,
            0.5,
            f"IMAGE ERROR\n{class_name}\n{str(e)[:40]}",
            ha="center",
            va="center",
            fontsize=9
        )

        ax.axis("off")

# ------------------------------------------------
# 6. Hide unused subplot cells
# ------------------------------------------------

for j in range(
    len(visual_samples),
    len(axes)
):

    axes[j].axis("off")

# ------------------------------------------------
# 7. Main title
# ------------------------------------------------

fig.suptitle(
    "RANDOM VISUAL QUALITY CHECK — FINAL 4-CLASS DATASET",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.96]
)

# ------------------------------------------------
# 8. Save figure
# ------------------------------------------------

VISUAL_CHECK_PATH = os.path.join(
    RESULTS_DIR,
    "step12e_random_visual_quality_check.png"
)

plt.savefig(
    VISUAL_CHECK_PATH,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------
# 9. Final output
# ------------------------------------------------

print("\n" + "-" * 70)
print("VISUAL QUALITY CHECK COMPLETE")
print("-" * 70)

print("Images displayed:", len(visual_samples))
print("Samples per class:", SAMPLES_PER_CLASS)

print("\nSaved:")
print(VISUAL_CHECK_PATH)

print("\n" + "=" * 70)
print("STEP 12E COMPLETE")
print("=" * 70)

In [ ]:
# STEP 13 — CREATE TRAIN VALIDATION TEST SPLIT

from sklearn.model_selection import train_test_split
import pandas as pd
import os

print("=" * 70)
print("STEP 13 — TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)

# Load FINAL 4-class dataset INCLUDING MCC
FINAL_CLASSIFICATION_CSV = os.path.join(
    DATA_DIR,
    "final_4class_classification_dataset_with_mcc.csv"
)

final_df = pd.read_csv(
    FINAL_CLASSIFICATION_CSV
)

print("Total images:", len(final_df))

# ------------------------------------------------------------
# 1. First split: 70% Train + 30% Temporary
# ------------------------------------------------------------

train_df, temp_df = train_test_split(
    final_df,
    test_size=0.30,
    stratify=final_df["label"],
    random_state=SEED
)

# ------------------------------------------------------------
# 2. Second split: 15% Validation + 15% Test
# ------------------------------------------------------------

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

# ------------------------------------------------------------
# 3. Reset indexes
# ------------------------------------------------------------

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# ------------------------------------------------------------
# 4. Save splits
# ------------------------------------------------------------

TRAIN_CSV = os.path.join(
    DATA_DIR,
    "train.csv"
)

VAL_CSV = os.path.join(
    DATA_DIR,
    "val.csv"
)

TEST_CSV = os.path.join(
    DATA_DIR,
    "test.csv"
)

train_df.to_csv(
    TRAIN_CSV,
    index=False
)

val_df.to_csv(
    VAL_CSV,
    index=False
)

test_df.to_csv(
    TEST_CSV,
    index=False
)

# ------------------------------------------------------------
# 5. Display split sizes
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SPLIT SIZES")
print("-" * 70)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# ------------------------------------------------------------
# 6. Display class distributions
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("TRAIN CLASS DISTRIBUTION")
print("-" * 70)

print(
    train_df["class_name"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)

print("\n" + "-" * 70)
print("VALIDATION CLASS DISTRIBUTION")
print("-" * 70)

print(
    val_df["class_name"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)

print("\n" + "-" * 70)
print("TEST CLASS DISTRIBUTION")
print("-" * 70)

print(
    test_df["class_name"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)

# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SPLIT VERIFICATION")
print("-" * 70)

print(
    "Total after split:",
    len(train_df) + len(val_df) + len(test_df)
)

print(
    "Original total:",
    len(final_df)
)

print(
    "No data loss:",
    len(train_df) + len(val_df) + len(test_df) == len(final_df)
)

print("\nSaved:")
print(TRAIN_CSV)
print(VAL_CSV)
print(TEST_CSV)

print("\n" + "=" * 70)
print("STEP 13 COMPLETE")
print("=" * 70)

In [ ]:
# STEP 14 — CREATE IMAGE DATASET AND DATALOADERS

import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

print("=" * 70)
print("STEP 14 — IMAGE DATASET + DATALOADERS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Image settings
# ------------------------------------------------------------

IMAGE_SIZE = 224
BATCH_SIZE = 64

# ------------------------------------------------------------
# 2. Image transformations
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# 3. Custom dataset
# ------------------------------------------------------------

class SkinCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["image_path"]
        label = int(row["label"])

        image = Image.open(
            image_path
        ).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

# ------------------------------------------------------------
# 4. Create datasets
# ------------------------------------------------------------

train_dataset = SkinCancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinCancerDataset(
    val_df,
    transform=eval_transform
)

test_dataset = SkinCancerDataset(
    test_df,
    transform=eval_transform
)

# ------------------------------------------------------------
# 5. Create DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# 6. Verify one batch
# ------------------------------------------------------------

images, labels = next(iter(train_loader))

print("\nTrain dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

print("\nBatch image shape:", images.shape)
print("Batch label shape:", labels.shape)

print("\nFirst 10 labels:")
print(labels[:10].tolist())

print("\n" + "=" * 70)
print("STEP 14 COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 15 — EFFICIENTNET-B0 MODEL SETUP
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

print("=" * 70)
print("STEP 15 — EFFICIENTNET-B0 MODEL SETUP")
print("=" * 70)

# ------------------------------------------------------------
# 1. Select device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU count:",
        torch.cuda.device_count()
    )

    for i in range(torch.cuda.device_count()):
        print(
            "GPU",
            i,
            ":",
            torch.cuda.get_device_name(i)
        )

# ------------------------------------------------------------
# 2. Create EfficientNet-B0
# ------------------------------------------------------------

model = models.efficientnet_b0(
    weights=None
)

# ------------------------------------------------------------
# 3. Replace final classifier
# ------------------------------------------------------------

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)

# ------------------------------------------------------------
# 4. Use both T4 GPUs
# ------------------------------------------------------------

if torch.cuda.device_count() > 1:

    model = nn.DataParallel(model)

model = model.to(device)

# ------------------------------------------------------------
# 5. Model information
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nBackbone: EfficientNet-B0")
print("Input size:", IMAGE_SIZE)
print("Output classes:", NUM_CLASSES)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

print("\nModel architecture:")
print(model)

print("\nModel ready.")
print("=" * 70)
print("STEP 15 COMPLETE")
print("=" * 70)

In [ ]:
# STEP 16 — LOSS FUNCTION + OPTIMIZER

import torch
import torch.nn as nn
import torch.optim as optim

print("=" * 70)
print("STEP 16 — LOSS FUNCTION + OPTIMIZER")
print("=" * 70)

# ------------------------------------------------------------
# 1. Class counts from training data
# ------------------------------------------------------------

class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
)

print("Training class counts:")

for label, count in class_counts.items():
    print(f"Class {label} ({CLASS_NAMES[label]}): {count}")

# ------------------------------------------------------------
# 2. Stable class weights
# ------------------------------------------------------------

total_samples = len(train_df)

class_weights = (
    total_samples / (NUM_CLASSES * class_counts)
).pow(0.5)

class_weights = (
    class_weights / class_weights.mean()
)

print("\nClass weights:")

for label, weight in class_weights.items():
    print(
        f"{CLASS_NAMES[label]}: {weight:.4f}"
    )

# ------------------------------------------------------------
# 3. Move weights to GPU
# ------------------------------------------------------------

class_weights_tensor = torch.tensor(
    class_weights.values,
    dtype=torch.float32,
    device=device
)

# ------------------------------------------------------------
# 4. Cross Entropy Loss
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

# ------------------------------------------------------------
# 5. Adam optimizer
# ------------------------------------------------------------

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("\nLoss function:")
print("CrossEntropyLoss with class weights")

print("\nOptimizer:")
print("Adam")
print("Learning rate: 1e-4")
print("Weight decay: 1e-4")

print("\nModel:")
print("EfficientNet-B0")
print("Output classes:", NUM_CLASSES)

print("\n" + "=" * 70)
print("STEP 16 COMPLETE")
print("=" * 70)

In [ ]:
# STEP 17 — FINAL EFFICIENTNET-B0 TRAINING

import torch
import torch.nn as nn
import torch.optim as optim
import time
import os

print("=" * 70)
print("STEP 17 — FINAL EFFICIENTNET-B0 TRAINING")
print("=" * 70)

# --------------------------------------------------
# 1. DEVICE
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("GPU count:", torch.cuda.device_count())

# --------------------------------------------------
# 2. DATA LOADERS
# --------------------------------------------------

train_loader_final = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

train_eval_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

val_loader_final = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# --------------------------------------------------
# 3. EFFICIENTNET-B0
# --------------------------------------------------

model_final = models.efficientnet_b0(
    weights=None
)

# Replace final classifier
model_final.classifier[1] = nn.Linear(
    model_final.classifier[1].in_features,
    NUM_CLASSES
)

# Move to device
model_final = model_final.to(device)

# Multi-GPU
if torch.cuda.device_count() > 1:
    model_final = nn.DataParallel(model_final)

print("Backbone: EfficientNet-B0")
print("Initialization: From scratch")
print("Output classes:", NUM_CLASSES)

# --------------------------------------------------
# 4. CLASS WEIGHTS
# --------------------------------------------------

class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
    .values
)

class_weights = 1.0 / torch.sqrt(
    torch.tensor(
        class_counts,
        dtype=torch.float32
    )
)

class_weights = (
    class_weights / class_weights.mean()
).to(device)

print("\nClass weights:")

for i, name in enumerate(CLASS_NAMES):
    print(
        f"{name:<12}: "
        f"{class_weights[i].item():.4f}"
    )

# --------------------------------------------------
# 5. LOSS
# --------------------------------------------------

criterion_final = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

# --------------------------------------------------
# 6. OPTIMIZER
# --------------------------------------------------

optimizer_final = optim.AdamW(
    model_final.parameters(),
    lr=5e-5,
    weight_decay=5e-4
)

scheduler_final = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_final,
    mode="min",
    factor=0.5,
    patience=1
)

# --------------------------------------------------
# 7. HISTORY
# --------------------------------------------------

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

best_val_loss = float("inf")
best_val_acc = 0.0
best_epoch = 0

# --------------------------------------------------
# 8. CHECKPOINT PATH
# --------------------------------------------------

checkpoint_final = (
    "/kaggle/working/skin_cancer_project/"
    "checkpoints/skin_cancer_efficientnetb0_FINAL_best.pth"
)

os.makedirs(
    os.path.dirname(checkpoint_final),
    exist_ok=True
)

NUM_EPOCHS = 10

# --------------------------------------------------
# 9. TRAINING LOOP
# --------------------------------------------------

for epoch in range(NUM_EPOCHS):

    start_time = time.time()

    # ==================================================
    # TRAINING
    # ==================================================

    model_final.train()

    running_loss = 0.0
    total = 0

    for images, labels in train_loader_final:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_final.zero_grad(
            set_to_none=True
        )

        outputs = model_final(images)

        loss = criterion_final(
            outputs,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model_final.parameters(),
            max_norm=1.0
        )

        optimizer_final.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        total += labels.size(0)

    # ==================================================
    # TRAINING METRICS
    # ==================================================

    model_final.eval()

    train_eval_loss = 0.0
    train_correct = 0
    train_total = 0

    with torch.no_grad():

        for images, labels in train_eval_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_final(images)

            loss = criterion_final(
                outputs,
                labels
            )

            train_eval_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            train_correct += (
                predictions == labels
            ).sum().item()

            train_total += labels.size(0)

    train_loss = (
        train_eval_loss / train_total
    )

    train_acc = (
        train_correct / train_total
    )

    # ==================================================
    # VALIDATION
    # ==================================================

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader_final:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_final(images)

            loss = criterion_final(
                outputs,
                labels
            )

            val_loss_sum += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)

    val_loss = (
        val_loss_sum / val_total
    )

    val_acc = (
        val_correct / val_total
    )

    # ==================================================
    # SAVE HISTORY
    # ==================================================

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["train_acc"].append(
        train_acc
    )

    history["val_acc"].append(
        val_acc
    )

    # ==================================================
    # LEARNING RATE SCHEDULER
    # ==================================================

    scheduler_final.step(
        val_loss
    )

    current_lr = optimizer_final.param_groups[0]["lr"]

    elapsed = time.time() - start_time

    # ==================================================
    # EPOCH RESULT
    # ==================================================

    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}% | "
        f"LR: {current_lr:.2e} | "
        f"Time: {elapsed/60:.2f} min"
    )

    # ==================================================
    # SAVE BEST MODEL
    # ==================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_val_acc = val_acc
        best_epoch = epoch + 1

        torch.save(
            {
                "epoch": epoch + 1,

                "model_state_dict":
                    model_final.state_dict(),

                "optimizer_state_dict":
                    optimizer_final.state_dict(),

                "scheduler_state_dict":
                    scheduler_final.state_dict(),

                "val_loss":
                    val_loss,

                "val_acc":
                    val_acc,

                "class_names":
                    CLASS_NAMES,

                "num_classes":
                    NUM_CLASSES,

                "model_name":
                    "EfficientNet-B0",

                "input_size":
                    224
            },
            checkpoint_final
        )

        print(
            "   -> Best EfficientNet-B0 model saved"
        )

# --------------------------------------------------
# 10. FINAL SUMMARY
# --------------------------------------------------

print("\n" + "=" * 70)
print("STEP 17 COMPLETE")
print("=" * 70)

print(f"Model         : EfficientNet-B0")
print(f"Best Epoch    : {best_epoch}")
print(f"Best Val Loss : {best_val_loss:.4f}")
print(f"Best Val Acc  : {best_val_acc*100:.2f}%")
print(f"Checkpoint    : {checkpoint_final}")

print("=" * 70)

In [ ]:
# ============================================================
# STEP 17A — FINAL EFFICIENTNET-B0 CHECKPOINT SAVE + BACKUP
# ============================================================

import os
import shutil
import json
from datetime import datetime
import torch

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_DIR = "/kaggle/working/skin_cancer_project"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
BACKUP_DIR = os.path.join(PROJECT_DIR, "backups")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "skin_cancer_efficientnetb0_FINAL_best.pth"
)

BACKUP_PATH = os.path.join(
    BACKUP_DIR,
    "skin_cancer_efficientnetb0_FINAL_best_backup.pth"
)

METADATA_PATH = os.path.join(
    BACKUP_DIR,
    "efficientnetb0_checkpoint_metadata.json"
)

# ------------------------------------------------------------
# 2. Check original checkpoint
# ------------------------------------------------------------

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"❌ EfficientNet-B0 checkpoint not found:\n{CHECKPOINT_PATH}\n\n"
        "Please run STEP 17 first."
    )

checkpoint_size_mb = os.path.getsize(CHECKPOINT_PATH) / (1024 ** 2)

print("=" * 70)
print("STEP 17A — EFFICIENTNET-B0 CHECKPOINT BACKUP")
print("=" * 70)

print(f"Original checkpoint:")
print(CHECKPOINT_PATH)

print(f"Checkpoint size: {checkpoint_size_mb:.2f} MB")

# ------------------------------------------------------------
# 3. Load and verify checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print("\nCheckpoint verified successfully.")

print("\nCheckpoint information:")

if "epoch" in checkpoint:
    print("Best epoch:", checkpoint["epoch"])

if "val_loss" in checkpoint:
    print("Best validation loss:", checkpoint["val_loss"])

if "val_acc" in checkpoint:
    print("Best validation accuracy:", checkpoint["val_acc"])

if "model_name" in checkpoint:
    print("Model:", checkpoint["model_name"])

if "num_classes" in checkpoint:
    print("Classes:", checkpoint["num_classes"])

if "class_names" in checkpoint:
    print("Class names:", checkpoint["class_names"])

if "input_size" in checkpoint:
    print("Input size:", checkpoint["input_size"])

# ------------------------------------------------------------
# 4. Confirm EfficientNet-B0
# ------------------------------------------------------------

model_name = checkpoint.get("model_name", "Unknown")

if model_name != "EfficientNet-B0":
    raise ValueError(
        f"❌ Wrong model checkpoint detected: {model_name}\n"
        "Expected: EfficientNet-B0"
    )

print("\n✅ Model architecture confirmed: EfficientNet-B0")

# ------------------------------------------------------------
# 5. Create backup copy
# ------------------------------------------------------------

shutil.copy2(
    CHECKPOINT_PATH,
    BACKUP_PATH
)

print("\nBackup created:")
print(BACKUP_PATH)

# ------------------------------------------------------------
# 6. Verify backup
# ------------------------------------------------------------

if not os.path.exists(BACKUP_PATH):
    raise FileNotFoundError(
        "❌ Backup file was not created."
    )

backup_size_mb = os.path.getsize(BACKUP_PATH) / (1024 ** 2)

if os.path.getsize(CHECKPOINT_PATH) != os.path.getsize(BACKUP_PATH):
    raise RuntimeError(
        "❌ Original checkpoint and backup sizes do not match."
    )

print(f"Backup size: {backup_size_mb:.2f} MB")
print("✅ Backup integrity check passed.")

# ------------------------------------------------------------
# 7. Save metadata
# ------------------------------------------------------------

metadata = {
    "model_name": "EfficientNet-B0",
    "checkpoint_file": os.path.basename(CHECKPOINT_PATH),
    "backup_file": os.path.basename(BACKUP_PATH),
    "class_names": checkpoint.get(
        "class_names",
        ["Melanoma", "BCC", "SCC", "MCC"]
    ),
    "num_classes": checkpoint.get("num_classes", 4),
    "input_size": checkpoint.get("input_size", 224),
    "framework": "PyTorch",
    "device_used": str(
        torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ),
    "gpu_count": torch.cuda.device_count(),
    "created_at": datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
    "best_epoch": checkpoint.get("epoch", None),
    "best_val_loss": checkpoint.get("val_loss", None),
    "best_val_accuracy": checkpoint.get("val_acc", None)
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=4
    )

print("\nMetadata saved:")
print(METADATA_PATH)

# ------------------------------------------------------------
# 8. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ STEP 17A COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nFINAL FILES")
print("-" * 70)

print("1. Main checkpoint:")
print(CHECKPOINT_PATH)

print("\n2. Backup checkpoint:")
print(BACKUP_PATH)

print("\n3. Metadata:")
print(METADATA_PATH)

print("\nModel: EfficientNet-B0")
print("Classes: Melanoma | BCC | SCC | MCC")
print("Input size: 224 × 224")

print("\n⚠️ IMPORTANT:")
print("Save a Kaggle Version with output files enabled")
print("to make the checkpoint persistent across sessions.")

In [ ]:
# ============================================================
# CHECK — FIND EFFICIENTNET-B0 CHECKPOINT
# ============================================================

import os

PROJECT_DIR = "/kaggle/working/skin_cancer_project"

print("=" * 70)
print("CHECKING EFFICIENTNET-B0 CHECKPOINT")
print("=" * 70)

print("\nProject directory exists:",
      os.path.exists(PROJECT_DIR))

print("\nAll .pth files under /kaggle/working:\n")

pth_found = []

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.lower().endswith(".pth"):
            full_path = os.path.join(root, file)
            size_mb = os.path.getsize(full_path) / (1024 ** 2)
            pth_found.append((full_path, size_mb))

if len(pth_found) == 0:
    print("❌ NO .pth FILE FOUND")
else:
    for path, size in pth_found:
        print(f"✅ {size:.2f} MB")
        print(path)
        print()

print("=" * 70)

# Specifically check our expected checkpoint
EXPECTED = (
    "/kaggle/working/skin_cancer_project/"
    "checkpoints/skin_cancer_efficientnetb0_FINAL_best.pth"
)

print("\nExpected checkpoint:")
print(EXPECTED)

if os.path.exists(EXPECTED):
    print("\n✅ CHECKPOINT EXISTS!")
    print(
        f"Size: "
        f"{os.path.getsize(EXPECTED)/(1024**2):.2f} MB"
    )
else:
    print("\n❌ CHECKPOINT DOES NOT EXIST IN CURRENT SESSION.")

In [ ]:
# ============================================================
# STEP 18 — FINAL EFFICIENTNET-B0 TEST EVALUATION
# ============================================================

import os
import json
import torch
import torch.nn as nn
import pandas as pd

from torch.utils.data import DataLoader
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=" * 70)
print("STEP 18 — FINAL EFFICIENTNET-B0 TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Paths and settings
# ------------------------------------------------------------

checkpoint_path = (
    "/kaggle/working/skin_cancer_project/"
    "checkpoints/skin_cancer_efficientnetb0_FINAL_best.pth"
)

results_dir = (
    "/kaggle/working/skin_cancer_project/results"
)

os.makedirs(results_dir, exist_ok=True)

CLASS_NAMES = [
    "Melanoma",
    "BCC",
    "SCC",
    "MCC"
]

NUM_CLASSES = 4
INPUT_SIZE = 224

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model: EfficientNet-B0")
print("Input size:", INPUT_SIZE)
print("Number of classes:", NUM_CLASSES)

# ------------------------------------------------------------
# 2. Check test dataset
# ------------------------------------------------------------

if "test_dataset" not in globals():
    raise NameError(
        "test_dataset was not found.\n"
        "Please run the dataset preparation / STEP 14 cell "
        "before STEP 18."
    )

print("\nTest dataset found.")
print("Test samples:", len(test_dataset))

# ------------------------------------------------------------
# 3. Create safe test DataLoader
# ------------------------------------------------------------

safe_test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Test DataLoader ready.")

# ------------------------------------------------------------
# 4. Check EfficientNet-B0 checkpoint
# ------------------------------------------------------------

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        f"\n❌ EfficientNet-B0 checkpoint not found:\n"
        f"{checkpoint_path}\n\n"
        "Please make sure STEP 17 and STEP 17A completed successfully."
    )

print("\nCheckpoint found:")
print(checkpoint_path)

# ------------------------------------------------------------
# 5. Load checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

print("\nCheckpoint loaded successfully.")

# ------------------------------------------------------------
# 6. Verify checkpoint model
# ------------------------------------------------------------

if isinstance(checkpoint, dict):

    checkpoint_model_name = checkpoint.get(
        "model_name",
        "Unknown"
    )

    checkpoint_num_classes = checkpoint.get(
        "num_classes",
        NUM_CLASSES
    )

    checkpoint_input_size = checkpoint.get(
        "input_size",
        INPUT_SIZE
    )

    checkpoint_epoch = checkpoint.get(
        "epoch",
        None
    )

    checkpoint_val_loss = checkpoint.get(
        "val_loss",
        None
    )

    checkpoint_val_acc = checkpoint.get(
        "val_acc",
        None
    )

else:

    checkpoint_model_name = "Unknown"
    checkpoint_num_classes = NUM_CLASSES
    checkpoint_input_size = INPUT_SIZE
    checkpoint_epoch = None
    checkpoint_val_loss = None
    checkpoint_val_acc = None


print("\nCheckpoint information:")
print("Model:", checkpoint_model_name)
print("Classes:", checkpoint_num_classes)
print("Input size:", checkpoint_input_size)

if checkpoint_epoch is not None:
    print("Best epoch:", checkpoint_epoch)

if checkpoint_val_loss is not None:
    print(
        f"Best validation loss: "
        f"{checkpoint_val_loss:.4f}"
    )

if checkpoint_val_acc is not None:
    print(
        f"Best validation accuracy: "
        f"{checkpoint_val_acc * 100:.2f}%"
    )


if checkpoint_model_name != "EfficientNet-B0":
    raise ValueError(
        f"\n❌ Wrong checkpoint model: "
        f"{checkpoint_model_name}\n"
        "Expected: EfficientNet-B0"
    )

if checkpoint_num_classes != NUM_CLASSES:
    raise ValueError(
        f"\n❌ Wrong number of classes: "
        f"{checkpoint_num_classes}\n"
        f"Expected: {NUM_CLASSES}"
    )

print("\n✅ Checkpoint architecture verified.")

# ------------------------------------------------------------
# 7. Rebuild EfficientNet-B0
# ------------------------------------------------------------

print("\nRebuilding EfficientNet-B0...")

model = models.efficientnet_b0(
    weights=None
)

# Replace final classifier
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)

# ------------------------------------------------------------
# 8. Extract model state dictionary
# ------------------------------------------------------------

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:

    state_dict = checkpoint["model_state_dict"]

elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:

    state_dict = checkpoint["state_dict"]

else:

    state_dict = checkpoint

# ------------------------------------------------------------
# 9. Remove DataParallel prefix if present
# ------------------------------------------------------------

clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):
        new_key = key.replace(
            "module.",
            "",
            1
        )
    else:
        new_key = key

    clean_state_dict[new_key] = value

# ------------------------------------------------------------
# 10. Load weights
# ------------------------------------------------------------

load_result = model.load_state_dict(
    clean_state_dict,
    strict=True
)

print("\nEfficientNet-B0 weights loaded successfully.")
print(load_result)

# ------------------------------------------------------------
# 11. Move model to device
# ------------------------------------------------------------

model = model.to(device)

model.eval()

print("Model moved to:", device)
print("Model set to evaluation mode.")

# ------------------------------------------------------------
# 12. Test evaluation
# ------------------------------------------------------------

print("\nStarting official test evaluation...")

all_labels = []
all_predictions = []

with torch.no_grad():

    for batch_idx, (images, labels) in enumerate(
        safe_test_loader
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_labels.extend(
            labels.cpu().numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

print("Test inference completed.")

# ------------------------------------------------------------
# 13. Sanity check
# ------------------------------------------------------------

if len(all_labels) != len(test_dataset):

    raise RuntimeError(
        f"❌ Prediction count mismatch.\n"
        f"Expected: {len(test_dataset)}\n"
        f"Got: {len(all_labels)}"
    )

print(
    f"Prediction count verified: "
    f"{len(all_predictions)}"
)

# ------------------------------------------------------------
# 14. Overall metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )
)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )
)

# ------------------------------------------------------------
# 15. Print final test results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL EFFICIENTNET-B0 TEST RESULTS")
print("=" * 70)

print(
    f"Test Samples        : {len(all_labels)}"
)

print(
    f"Test Accuracy       : "
    f"{accuracy * 100:.2f}%"
)

print(
    f"Weighted Precision  : "
    f"{weighted_precision * 100:.2f}%"
)

print(
    f"Weighted Recall     : "
    f"{weighted_recall * 100:.2f}%"
)

print(
    f"Weighted F1         : "
    f"{weighted_f1 * 100:.2f}%"
)

print(
    f"Macro Precision     : "
    f"{macro_precision * 100:.2f}%"
)

print(
    f"Macro Recall        : "
    f"{macro_recall * 100:.2f}%"
)

print(
    f"Macro F1            : "
    f"{macro_f1 * 100:.2f}%"
)

# ------------------------------------------------------------
# 16. Classification report
# ------------------------------------------------------------

report = classification_report(
    all_labels,
    all_predictions,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

print("\n" + "=" * 70)
print("CLASS-WISE PERFORMANCE")
print("=" * 70)

for class_name in CLASS_NAMES:

    print(
        f"{class_name:10s} | "
        f"Precision: "
        f"{report[class_name]['precision'] * 100:6.2f}% | "
        f"Recall: "
        f"{report[class_name]['recall'] * 100:6.2f}% | "
        f"F1: "
        f"{report[class_name]['f1-score'] * 100:6.2f}% | "
        f"Support: "
        f"{int(report[class_name]['support'])}"
    )

# ------------------------------------------------------------
# 17. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=list(range(NUM_CLASSES))
)

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"Actual {x}"
        for x in CLASS_NAMES
    ],
    columns=[
        f"Pred {x}"
        for x in CLASS_NAMES
    ]
)

print(cm_df)

# ------------------------------------------------------------
# 18. Save final overall metrics
# ------------------------------------------------------------

final_metrics = {

    "model_name": "EfficientNet-B0",

    "checkpoint": checkpoint_path,

    "checkpoint_epoch": checkpoint_epoch,

    "checkpoint_val_loss": (
        float(checkpoint_val_loss)
        if checkpoint_val_loss is not None
        else None
    ),

    "checkpoint_val_accuracy": (
        float(checkpoint_val_acc)
        if checkpoint_val_acc is not None
        else None
    ),

    "test_samples": len(all_labels),

    "test_accuracy": float(accuracy),

    "weighted_precision": float(
        weighted_precision
    ),

    "weighted_recall": float(
        weighted_recall
    ),

    "weighted_f1": float(
        weighted_f1
    ),

    "macro_precision": float(
        macro_precision
    ),

    "macro_recall": float(
        macro_recall
    ),

    "macro_f1": float(
        macro_f1
    ),

    "class_names": CLASS_NAMES,

    "input_size": INPUT_SIZE
}

metrics_path = os.path.join(
    results_dir,
    "efficientnetb0_final_test_metrics.json"
)

with open(
    metrics_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_metrics,
        f,
        indent=4
    )

print("\nOverall metrics saved:")
print(metrics_path)

# ------------------------------------------------------------
# 19. Save class-wise results
# ------------------------------------------------------------

class_rows = []

for class_name in CLASS_NAMES:

    class_rows.append({

        "Class": class_name,

        "Precision": report[class_name][
            "precision"
        ],

        "Recall": report[class_name][
            "recall"
        ],

        "F1": report[class_name][
            "f1-score"
        ],

        "Support": int(
            report[class_name][
                "support"
            ]
        )
    })

class_df = pd.DataFrame(
    class_rows
)

class_results_path = os.path.join(
    results_dir,
    "efficientnetb0_class_wise_results.csv"
)

class_df.to_csv(
    class_results_path,
    index=False
)

print("Class-wise results saved:")
print(class_results_path)

# ------------------------------------------------------------
# 20. Save confusion matrix
# ------------------------------------------------------------

cm_results_path = os.path.join(
    results_dir,
    "efficientnetb0_confusion_matrix.csv"
)

cm_df.to_csv(
    cm_results_path
)

print("Confusion matrix saved:")
print(cm_results_path)

# ------------------------------------------------------------
# 21. Save raw predictions
# ------------------------------------------------------------

prediction_df = pd.DataFrame({

    "Actual_Label_ID": all_labels,

    "Predicted_Label_ID": all_predictions,

    "Actual_Class": [
        CLASS_NAMES[x]
        for x in all_labels
    ],

    "Predicted_Class": [
        CLASS_NAMES[x]
        for x in all_predictions
    ]
})

prediction_path = os.path.join(
    results_dir,
    "efficientnetb0_test_predictions.csv"
)

prediction_df.to_csv(
    prediction_path,
    index=False
)

print("Raw predictions saved:")
print(prediction_path)

# ------------------------------------------------------------
# 22. MCC warning
# ------------------------------------------------------------

mcc_support = int(
    report["MCC"]["support"]
)

print("\n" + "=" * 70)
print("IMPORTANT CLASS SUPPORT NOTE")
print("=" * 70)

print(
    f"MCC test support: {mcc_support} images"
)

if mcc_support < 20:

    print(
        "⚠️ MCC performance should be interpreted "
        "with caution because the official test "
        "sample size is small."
    )

# ------------------------------------------------------------
# 23. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ STEP 18 COMPLETE")
print("=" * 70)

print("Model       : EfficientNet-B0")
print(
    f"Test Acc    : {accuracy * 100:.2f}%"
)
print(
    f"Macro F1    : {macro_f1 * 100:.2f}%"
)
print(
    f"Weighted F1 : {weighted_f1 * 100:.2f}%"
)

print("\nResults directory:")
print(results_dir)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 19 — FINAL EFFICIENTNET-B0 TRAINING & VALIDATION CURVES
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("STEP 19 — EFFICIENTNET-B0 TRAINING & VALIDATION CURVES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check training history
# ------------------------------------------------------------

if "history" not in globals():
    raise NameError(
        "Training history was not found. "
        "Please run STEP 17 first."
    )

required_keys = [
    "train_loss",
    "val_loss",
    "train_acc",
    "val_acc"
]

for key in required_keys:
    if key not in history:
        raise KeyError(
            f"'{key}' not found in training history."
        )

# ------------------------------------------------------------
# 2. Results directory
# ------------------------------------------------------------

RESULTS_DIR = (
    "/kaggle/working/skin_cancer_project/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------------------------------------
# 3. Create history DataFrame
# ------------------------------------------------------------

history_df = pd.DataFrame(history)

history_df.insert(
    0,
    "Epoch",
    range(1, len(history_df) + 1)
)

epochs = history_df["Epoch"]

# ------------------------------------------------------------
# 4. Find best epochs
# ------------------------------------------------------------

best_loss_idx = history_df["val_loss"].idxmin()
best_loss_epoch = int(
    history_df.loc[best_loss_idx, "Epoch"]
)

best_acc_idx = history_df["val_acc"].idxmax()
best_acc_epoch = int(
    history_df.loc[best_acc_idx, "Epoch"]
)

best_val_loss = float(
    history_df.loc[best_loss_idx, "val_loss"]
)

best_val_acc = float(
    history_df.loc[best_acc_idx, "val_acc"]
)

print("\nTraining epochs:", len(history_df))

print(
    f"Best validation loss : "
    f"Epoch {best_loss_epoch} "
    f"({best_val_loss:.4f})"
)

print(
    f"Best validation accuracy : "
    f"Epoch {best_acc_epoch} "
    f"({best_val_acc:.2f}%)"
)

# ------------------------------------------------------------
# 5. LOSS CURVE
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    epochs,
    history_df["train_loss"],
    marker="o",
    linewidth=2,
    label="Training Loss"
)

plt.plot(
    epochs,
    history_df["val_loss"],
    marker="o",
    linewidth=2,
    label="Validation Loss"
)

plt.axvline(
    best_loss_epoch,
    linestyle="--",
    linewidth=1.5,
    label=f"Best Val Loss — Epoch {best_loss_epoch}"
)

plt.title(
    "EfficientNet-B0 Training and Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.xticks(
    list(epochs)
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()

loss_plot_path = os.path.join(
    RESULTS_DIR,
    "efficientnetb0_training_validation_loss.png"
)

plt.savefig(
    loss_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "\nLoss curve saved:"
)
print(loss_plot_path)

# ------------------------------------------------------------
# 6. ACCURACY CURVE
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    epochs,
    history_df["train_acc"],
    marker="o",
    linewidth=2,
    label="Training Accuracy"
)

plt.plot(
    epochs,
    history_df["val_acc"],
    marker="o",
    linewidth=2,
    label="Validation Accuracy"
)

plt.axvline(
    best_acc_epoch,
    linestyle="--",
    linewidth=1.5,
    label=f"Best Val Accuracy — Epoch {best_acc_epoch}"
)

plt.title(
    "EfficientNet-B0 Training and Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")

plt.xticks(
    list(epochs)
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()

accuracy_plot_path = os.path.join(
    RESULTS_DIR,
    "efficientnetb0_training_validation_accuracy.png"
)

plt.savefig(
    accuracy_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "\nAccuracy curve saved:"
)
print(accuracy_plot_path)

# ------------------------------------------------------------
# 7. SAVE TRAINING HISTORY
# ------------------------------------------------------------

history_csv_path = os.path.join(
    RESULTS_DIR,
    "efficientnetb0_training_history.csv"
)

history_df.to_csv(
    history_csv_path,
    index=False
)

print(
    "\nTraining history saved:"
)
print(history_csv_path)

# ------------------------------------------------------------
# 8. DISPLAY HISTORY TABLE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING HISTORY")
print("=" * 70)

display(
    history_df.round(4)
)

# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 19 COMPLETE")
print("=" * 70)

print("Model: EfficientNet-B0")
print(f"Epochs: {len(history_df)}")

print(
    f"Best Val Loss: "
    f"{best_val_loss:.4f} "
    f"(Epoch {best_loss_epoch})"
)

print(
    f"Best Val Accuracy: "
    f"{best_val_acc:.2f}% "
    f"(Epoch {best_acc_epoch})"
)

print("\nSaved files:")
print(
    "1.",
    loss_plot_path
)
print(
    "2.",
    accuracy_plot_path
)
print(
    "3.",
    history_csv_path
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 19B — HIGHLIGHT TRAINING / VALIDATION CURVE CROSSINGS
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("STEP 19B — CURVE CROSSING ANALYSIS")
print("=" * 70)

RESULTS_DIR = "/kaggle/working/skin_cancer_project/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load history
history_df = pd.DataFrame(history)
history_df.insert(0, "Epoch", range(1, len(history_df) + 1))

epochs = history_df["Epoch"].values

# ------------------------------------------------------------
# Detect crossings
# ------------------------------------------------------------

loss_diff = (
    history_df["train_loss"].values -
    history_df["val_loss"].values
)

acc_diff = (
    history_df["train_acc"].values -
    history_df["val_acc"].values
)

loss_crossings = np.where(np.sign(loss_diff[:-1]) != np.sign(loss_diff[1:]))[0]
acc_crossings = np.where(np.sign(acc_diff[:-1]) != np.sign(acc_diff[1:]))[0]

print("\nLoss curve crossings:")
if len(loss_crossings) == 0:
    print("No crossing detected.")
else:
    for i in loss_crossings:
        print(f"  Between Epoch {epochs[i]} and Epoch {epochs[i+1]}")

print("\nAccuracy curve crossings:")
if len(acc_crossings) == 0:
    print("No crossing detected.")
else:
    for i in acc_crossings:
        print(f"  Between Epoch {epochs[i]} and Epoch {epochs[i+1]}")

# ------------------------------------------------------------
# LOSS CURVE
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    epochs,
    history_df["train_loss"],
    marker="o",
    linewidth=2,
    label="Training Loss"
)

plt.plot(
    epochs,
    history_df["val_loss"],
    marker="o",
    linewidth=2,
    label="Validation Loss"
)

# Highlight crossing region
for i in loss_crossings:
    x1 = epochs[i]
    x2 = epochs[i + 1]

    plt.axvspan(
        x1,
        x2,
        alpha=0.15,
        label="Curve Crossing Region"
        if i == loss_crossings[0]
        else None
    )

    plt.annotate(
        f"Crossing\nEpoch {x1}–{x2}",
        xy=((x1 + x2) / 2, 
            (history_df["train_loss"].iloc[i] +
             history_df["val_loss"].iloc[i]) / 2),
        xytext=(0, 25),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        arrowprops=dict(arrowstyle="->")
    )

plt.title("EfficientNet-B0 Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

loss_path = os.path.join(
    RESULTS_DIR,
    "efficientnetb0_loss_curve_crossing_highlighted.png"
)

plt.savefig(loss_path, dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# ACCURACY CURVE
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    epochs,
    history_df["train_acc"],
    marker="o",
    linewidth=2,
    label="Training Accuracy"
)

plt.plot(
    epochs,
    history_df["val_acc"],
    marker="o",
    linewidth=2,
    label="Validation Accuracy"
)

# Highlight crossing region
for i in acc_crossings:
    x1 = epochs[i]
    x2 = epochs[i + 1]

    plt.axvspan(
        x1,
        x2,
        alpha=0.15,
        label="Curve Crossing Region"
        if i == acc_crossings[0]
        else None
    )

    plt.annotate(
        f"Crossing\nEpoch {x1}–{x2}",
        xy=((x1 + x2) / 2,
            (history_df["train_acc"].iloc[i] +
             history_df["val_acc"].iloc[i]) / 2),
        xytext=(0, -35),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        arrowprops=dict(arrowstyle="->")
    )

plt.title("EfficientNet-B0 Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.xticks(epochs)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

accuracy_path = os.path.join(
    RESULTS_DIR,
    "efficientnetb0_accuracy_curve_crossing_highlighted.png"
)

plt.savefig(accuracy_path, dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CROSSING ANALYSIS SUMMARY")
print("=" * 70)

if len(loss_crossings) > 0:
    print("✓ Loss curves cross between:")
    for i in loss_crossings:
        print(f"  Epoch {epochs[i]} → Epoch {epochs[i+1]}")
else:
    print("✗ No loss crossing detected.")

if len(acc_crossings) > 0:
    print("\n✓ Accuracy curves cross between:")
    for i in acc_crossings:
        print(f"  Epoch {epochs[i]} → Epoch {epochs[i+1]}")
else:
    print("\n✗ No accuracy crossing detected.")

print("\nSaved:")
print(loss_path)
print(accuracy_path)

print("=" * 70)
print("STEP 19B COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# FIX — ACCURACY CURVE AS PERCENTAGE
# ============================================================

train_acc_percent = history_df["train_acc"] * 100
val_acc_percent = history_df["val_acc"] * 100

plt.figure(figsize=(10, 6))

plt.plot(
    epochs,
    train_acc_percent,
    marker="o",
    linewidth=2,
    label="Training Accuracy"
)

plt.plot(
    epochs,
    val_acc_percent,
    marker="o",
    linewidth=2,
    label="Validation Accuracy"
)

for i in acc_crossings:
    x1 = epochs[i]
    x2 = epochs[i + 1]

    plt.axvspan(
        x1,
        x2,
        alpha=0.15,
        label="Curve Crossing Region"
        if i == acc_crossings[0]
        else None
    )

    plt.annotate(
        f"Crossing\nEpoch {x1}–{x2}",
        xy=(
            (x1 + x2) / 2,
            (train_acc_percent.iloc[i] +
             val_acc_percent.iloc[i]) / 2
        ),
        xytext=(0, -35),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        arrowprops=dict(arrowstyle="->")
    )

plt.title("EfficientNet-B0 Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.xticks(epochs)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

accuracy_path = os.path.join(
    RESULTS_DIR,
    "efficientnetb0_accuracy_curve_crossing_highlighted.png"
)

plt.savefig(accuracy_path, dpi=300, bbox_inches="tight")
plt.show()

print("Accuracy curve saved:")
print(accuracy_path)

In [ ]:
# ======================================================================
# STEP 20 — FINAL EFFICIENTNET-B0 CONFUSION MATRIX + CLASS-WISE PERFORMANCE
# ======================================================================

print("=" * 75)
print("STEP 20 — FINAL EFFICIENTNET-B0 TEST EVALUATION")
print("=" * 75)

import os
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import models
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# ----------------------------------------------------------------------
# SETTINGS
# ----------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RESULTS_DIR = "/kaggle/working/skin_cancer_project/results"

CHECKPOINT_PATH = (
    "/kaggle/working/skin_cancer_project/"
    "checkpoints/skin_cancer_efficientnetb0_FINAL_best.pth"
)

CLASS_NAMES = ["Melanoma", "BCC", "SCC", "MCC"]
NUM_CLASSES = 4

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Device:", device)
print("Checkpoint:", CHECKPOINT_PATH)
print("Model: EfficientNet-B0")
print("Classes:", CLASS_NAMES)

# ----------------------------------------------------------------------
# CHECK FILE
# ----------------------------------------------------------------------

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"EfficientNet-B0 checkpoint not found:\n{CHECKPOINT_PATH}"
    )

# ----------------------------------------------------------------------
# LOAD CHECKPOINT
# ----------------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

print("\nCheckpoint loaded.")

# ----------------------------------------------------------------------
# REBUILD FINAL EFFICIENTNET-B0
# ----------------------------------------------------------------------

model = models.efficientnet_b0(weights=None)

# Replace ImageNet 1000-class classifier with our 4-class classifier
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)

# ----------------------------------------------------------------------
# GET STATE DICTIONARY
# ----------------------------------------------------------------------

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:

    state_dict = checkpoint["model_state_dict"]

    print("Checkpoint type: Full training checkpoint")

    if "epoch" in checkpoint:
        print("Best epoch:", checkpoint["epoch"])

    if "val_loss" in checkpoint:
        print("Best validation loss:", checkpoint["val_loss"])

    if "val_acc" in checkpoint:
        print("Best validation accuracy:",
              f"{checkpoint['val_acc'] * 100:.2f}%")

else:

    state_dict = checkpoint

    print("Checkpoint type: State dictionary")

# ----------------------------------------------------------------------
# REMOVE DataParallel "module." PREFIX IF PRESENT
# ----------------------------------------------------------------------

clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):
        key = key[len("module."):]

    clean_state_dict[key] = value

# ----------------------------------------------------------------------
# LOAD WEIGHTS
# ----------------------------------------------------------------------

load_result = model.load_state_dict(
    clean_state_dict,
    strict=True
)

print("\nEfficientNet-B0 checkpoint loaded successfully.")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

# ----------------------------------------------------------------------
# MULTI-GPU SUPPORT
# ----------------------------------------------------------------------

if torch.cuda.device_count() > 1:
    print(f"\nUsing {torch.cuda.device_count()} GPUs for evaluation.")
    model = torch.nn.DataParallel(model)

model = model.to(device)
model.eval()

# ----------------------------------------------------------------------
# TEST DATA CHECK
# ----------------------------------------------------------------------

if "test_loader" not in globals():
    raise NameError(
        "test_loader was not found. Please run the official test "
        "DataLoader setup before STEP 20."
    )

# ----------------------------------------------------------------------
# COLLECT TEST PREDICTIONS
# ----------------------------------------------------------------------

all_predictions = []
all_targets = []

print("\nEvaluating official test set...")

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            labels.cpu().numpy()
        )

all_predictions = np.array(all_predictions)
all_targets = np.array(all_targets)

print("Test samples evaluated:", len(all_targets))

# ----------------------------------------------------------------------
# CONFUSION MATRIX
# ----------------------------------------------------------------------

LABELS = list(range(NUM_CLASSES))

cm = confusion_matrix(
    all_targets,
    all_predictions,
    labels=LABELS
)

print("\n" + "=" * 65)
print("CONFUSION MATRIX")
print("=" * 65)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {c}" for c in CLASS_NAMES],
    columns=[f"Pred {c}" for c in CLASS_NAMES]
)

print(cm_df.to_string())

# ----------------------------------------------------------------------
# SAVE CONFUSION MATRIX CSV
# ----------------------------------------------------------------------

cm_csv_path = os.path.join(
    RESULTS_DIR,
    "efficientnetb0_confusion_matrix_step20.csv"
)

cm_df.to_csv(cm_csv_path)

# ----------------------------------------------------------------------
# CONFUSION MATRIX FIGURE
# ----------------------------------------------------------------------

plt.figure(figsize=(9, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    linewidths=0.5,
    cbar=True
)

plt.title(
    "EfficientNet-B0 — Confusion Matrix\n"
    "Official Test Set"
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.tight_layout()

cm_png_path = os.path.join(
    RESULTS_DIR,
    "efficientnetb0_confusion_matrix_step20.png"
)

plt.savefig(
    cm_png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ----------------------------------------------------------------------
# CLASS-WISE METRICS
# ----------------------------------------------------------------------

precision = precision_score(
    all_targets,
    all_predictions,
    labels=LABELS,
    average=None,
    zero_division=0
)

recall = recall_score(
    all_targets,
    all_predictions,
    labels=LABELS,
    average=None,
    zero_division=0
)

f1 = f1_score(
    all_targets,
    all_predictions,
    labels=LABELS,
    average=None,
    zero_division=0
)

support = np.bincount(
    all_targets,
    minlength=NUM_CLASSES
)

class_results = pd.DataFrame({

    "Class": CLASS_NAMES,

    "Precision (%)":
        precision * 100,

    "Recall (%)":
        recall * 100,

    "F1 Score (%)":
        f1 * 100,

    "Support":
        support
})

print("\n" + "=" * 65)
print("CLASS-WISE PERFORMANCE")
print("=" * 65)

print(
    class_results.to_string(
        index=False,
        formatters={
            "Precision (%)": "{:.2f}".format,
            "Recall (%)": "{:.2f}".format,
            "F1 Score (%)": "{:.2f}".format
        }
    )
)

# ----------------------------------------------------------------------
# SAVE CLASS-WISE RESULTS
# ----------------------------------------------------------------------

class_results_path = os.path.join(
    RESULTS_DIR,
    "efficientnetb0_class_wise_results_step20.csv"
)

class_results.to_csv(
    class_results_path,
    index=False
)

# ----------------------------------------------------------------------
# OVERALL METRICS
# ----------------------------------------------------------------------

accuracy = (
    np.mean(
        all_targets == all_predictions
    ) * 100
)

weighted_precision = (
    precision_score(
        all_targets,
        all_predictions,
        average="weighted",
        zero_division=0
    ) * 100
)

weighted_recall = (
    recall_score(
        all_targets,
        all_predictions,
        average="weighted",
        zero_division=0
    ) * 100
)

weighted_f1 = (
    f1_score(
        all_targets,
        all_predictions,
        average="weighted",
        zero_division=0
    ) * 100
)

macro_precision = (
    precision_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    ) * 100
)

macro_recall = (
    recall_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    ) * 100
)

macro_f1 = (
    f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    ) * 100
)

# ----------------------------------------------------------------------
# OVERALL OUTPUT
# ----------------------------------------------------------------------

print("\n" + "=" * 65)
print("OVERALL TEST PERFORMANCE")
print("=" * 65)

print(f"Accuracy           : {accuracy:.2f}%")
print(f"Weighted Precision : {weighted_precision:.2f}%")
print(f"Weighted Recall    : {weighted_recall:.2f}%")
print(f"Weighted F1 Score  : {weighted_f1:.2f}%")
print(f"Macro Precision    : {macro_precision:.2f}%")
print(f"Macro Recall       : {macro_recall:.2f}%")
print(f"Macro F1 Score     : {macro_f1:.2f}%")

# ----------------------------------------------------------------------
# SAVE METRICS JSON
# ----------------------------------------------------------------------

metrics = {
    "model": "EfficientNet-B0",
    "test_samples": int(len(all_targets)),
    "accuracy": float(accuracy),
    "weighted_precision": float(weighted_precision),
    "weighted_recall": float(weighted_recall),
    "weighted_f1": float(weighted_f1),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1)
}

metrics_path = os.path.join(
    RESULTS_DIR,
    "efficientnetb0_final_test_metrics_step20.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

# ----------------------------------------------------------------------
# FINAL
# ----------------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 20 COMPLETE — EFFICIENTNET-B0")
print("=" * 75)

print("\nSaved files:")

print("1. Confusion Matrix CSV:")
print(cm_csv_path)

print("\n2. Confusion Matrix PNG:")
print(cm_png_path)

print("\n3. Class-wise Results CSV:")
print(class_results_path)

print("\n4. Final Metrics JSON:")
print(metrics_path)

print("=" * 75)